# Knee OA: compact vision–language grading versus ResNet-18

Publication notebook for the Master Thesis Project by Sepideh Azadian.

Read `docs/REPRODUCING.md` before running. This notebook contains the full study pipeline. The completed 18 November 2026 result snapshot is distributed separately. This publication copy was prepared from the available complete source notebook; it is not represented as the exact final executed Kaggle notebook. See `docs/SOURCE_PROVENANCE.json` for source hashes and publication changes.

**Fresh run:** use a new output directory. **Resume:** set `resume_from` to the restored study directory; saved configuration is loaded automatically, and skip sections 5a–5c when the frozen protocol is already present.


## 1 Prepare Kaggle

1. Create a private Kaggle notebook and import this `.ipynb`.
2. Attach **shashwatwork/knee-osteoarthritis-dataset-with-severity** with its `train`, `val`, `test` and optional `auto_test` folders. No manual copying into notebook code is required.
3. Select a GPU accelerator. The implementation uses one GPU (`cuda:0`) and runs models sequentially. Enable Internet for initial model and dependency downloads, or attach a complete local Hugging Face snapshot and a wheelhouse.
4. Run the cells in order. Wait for NOTEBOOK READY before section 2. Setup requests a Python-kernel restart only if an updated package was already imported. The notebook keeps Kaggle's installed PyTorch/torchvision pair and does not install FlashAttention.
5. In section 2, this copy selects `mode='study'` for your current run. Use `mode='pilot'` to repeat the small development pilot. `data_root=''` discovers the folder containing the three retained splits. Set it explicitly only if discovery finds more than one candidate.

The model and dataset terms, provenance, and institutional secondary-data requirements follow the proposal. Seven-digit prefixes are treated as provisional groups by default. Only set `grouping_verified=True` after documenting an authoritative mapping. Until then all results are an engineering benchmark with prefix-based intervals.

The following is a **target dependency configuration**, not a claim of a completed Kaggle GPU validation. The exact installed environment is recorded in each run.


**Dependency repair for modern Kaggle images.** This implementation retains Transformers 4.51.3 and Hub 0.30.2. Some preinstalled Gradio/Gradio Client/Diffusers releases require Hub 1.x and cannot share that target stack. These packages are not imported by this notebook. The setup removes only those three unused packages whose installed Hub requirement conflicts, and only inside Kaggle. It also constrains Pillow to `>=10.4,<13`. It does not upgrade PyTorch or torchvision. To retain the removed packages for other work, use a separate session/environment.

After changing packages, setup continues automatically if the changed modules were not already imported. Otherwise it prints RESTART REQUIRED and pauses; restart the Python kernel and rerun setup. It checks the actual model-class imports and a small CUDA NF4 operation in a fresh interpreter. It validates the notebook dependency graph, including selected extras. A global `pip check` report is retained; unrelated preinstalled-package conflicts such as Dopamine/Gym are logged without blocking this notebook. Import failures and broken requirements used by the notebook still stop execution. Do not upgrade Hub to 1.x while retaining this Transformers 4.x implementation.


**CUDA 13 runtime update.** The reported Kaggle runtime uses Python 3.13 and PyTorch with CUDA 13.0. The setup now targets bitsandbytes 0.49.2, whose official build list includes CUDA 13.0, instead of the original 0.45.5 pin. The small NF4 GPU check does not establish full-model memory fit or adapter feasibility; those remain the development pilot's job. If a formal experiment was already frozen under another backend/environment, start a separately documented run rather than mixing cached predictions.


**Successful installation is not an exception.** The previous unconditional `SystemExit: SETUP COMPLETE` has been removed. The setup now prints **NOTEBOOK READY** after import and notebook-dependency checks pass. Dopamine/Gym warnings remain logged as unrelated environment conflicts. If setup prints **RESTART REQUIRED**, it lists the changed modules still loaded in the Python process; section 2 is guarded until setup passes. If the same packages are downloaded again after every restart, the runtime may have been recreated; avoid repeatedly stopping and creating the whole session when only the Python kernel needs restarting.


In [ ]:
# Run this setup cell first. Continue when it prints NOTEBOOK READY.
SETUP_READY = False

def prepare_notebook_environment():
    import sys, subprocess, textwrap, importlib.metadata as metadata
    from pathlib import Path
    from packaging.requirements import Requirement
    from packaging.specifiers import SpecifierSet
    from packaging.utils import canonicalize_name

    if globals().get('_THESIS_SETUP_RESTART_MODULES'):
        print('RESTART REQUIRED: old imports remain in this Python process:', globals()['_THESIS_SETUP_RESTART_MODULES'])
        print('Restart the Python kernel, then rerun setup. Re-running this cell alone cannot clear old imports.')
        return False

    INSTALL_DEPENDENCIES = True
    REMOVE_UNUSED_CONFLICTING_PACKAGES = True  # Kaggle session only; explained above.
    WHEELHOUSE = ''  # Optional offline wheels; leave blank for normal downloads.
    TARGET = {
        'transformers': '4.51.3',
        'peft': '0.15.2',
        'accelerate': '1.6.0',
        'bitsandbytes': '0.49.2',
        'tokenizers': '0.21.1',
        'huggingface-hub': '0.30.2',
    }
    PILLOW_RANGE = '>=10.4,<13'

    def installed_version(package):
        try: return metadata.version(package)
        except metadata.PackageNotFoundError: return None

    # Transformers 4.51.3 needs Hub <1.0; upgrading Hub alone cannot satisfy the
    # modern Gradio/Diffusers requirements shown in the reported error.
    # Remove only these unused packages if their Hub requirement excludes our pin.
    conflicting_unused=[]
    for package in ['gradio','gradio-client','diffusers']:
        if installed_version(package) is None: continue
        for text in metadata.requires(package) or []:
            requirement=Requirement(text)
            active=requirement.marker is None or requirement.marker.evaluate({'extra':''})
            if (active and canonicalize_name(requirement.name)=='huggingface-hub'
                    and not requirement.specifier.contains(TARGET['huggingface-hub'])):
                conflicting_unused.append(package)
                break

    need=[f'{name}=={version}' for name,version in TARGET.items()
          if installed_version(name)!=version]
    pillow_version=installed_version('Pillow')
    if pillow_version is None or not SpecifierSet(PILLOW_RANGE).contains(pillow_version):
        need.append('Pillow'+PILLOW_RANGE)

    import_roots={
        'transformers':'transformers','peft':'peft','accelerate':'accelerate',
        'bitsandbytes':'bitsandbytes','tokenizers':'tokenizers',
        'huggingface-hub':'huggingface_hub','pillow':'PIL',
    }
    changed_roots={import_roots[canonicalize_name(Requirement(spec).name)] for spec in need}
    loaded_changed=sorted(root for root in changed_roots
        if any(name==root or name.startswith(root+'.') for name in sys.modules))
    for package,root in import_roots.items():
        module=sys.modules.get(root)
        loaded_version=getattr(module,'__version__',None) if module is not None else None
        if loaded_version is not None and loaded_version!=installed_version(package):
            loaded_changed.append(root)
    loaded_changed=sorted(set(loaded_changed))

    if conflicting_unused or need:
        if not INSTALL_DEPENDENCIES:
            raise RuntimeError('Setup needs changes. Enable INSTALL_DEPENDENCIES in a disposable Kaggle session.')
        if conflicting_unused:
            if not REMOVE_UNUSED_CONFLICTING_PACKAGES:
                raise RuntimeError('Incompatible unused packages: '+', '.join(conflicting_unused)+
                    '. Enable removal for this Kaggle session, or use a separate environment.')
            if not Path('/kaggle/working').is_dir():
                raise RuntimeError('Automatic removal is limited to Kaggle. Use an isolated environment locally.')
            print('Removing unused packages from this Kaggle session:',', '.join(conflicting_unused))
            subprocess.check_call([sys.executable,'-m','pip','uninstall','-y',*conflicting_unused])
        command=[sys.executable,'-m','pip','install']
        if WHEELHOUSE: command += ['--no-index','--find-links',WHEELHOUSE]
        # Pass the entire target set in one transaction so pip resolves them together.
        subprocess.check_call(command+[f'{name}=={version}' for name,version in TARGET.items()]
                              +['Pillow'+PILLOW_RANGE])
        if loaded_changed:
            globals()['_THESIS_SETUP_RESTART_MODULES']=loaded_changed
            print('RESTART REQUIRED: these updated modules were already imported:', ', '.join(loaded_changed))
            print('Restart the Python kernel once, then run this setup cell again before continuing.')
            print('No installation error occurred. The kernel restart is needed to clear old imports.')
            return False
        print('Installation complete. Updated modules were not already imported; continuing checks without a restart.')

    if loaded_changed:
        globals()['_THESIS_SETUP_RESTART_MODULES']=loaded_changed
        print('RESTART REQUIRED: loaded module versions differ from installed packages:', ', '.join(loaded_changed))
        return False

    # A pip resolver message is not an import test. Test the actual notebook imports
    # in a fresh child interpreter; do not download/load model weights here.
    probe = """
    import torch, torchvision, transformers, peft, accelerate, bitsandbytes, PIL
    from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
    # Check the installed GPU quantization backend with a tiny layer before any
    # model download. The actual multimodal inference/training pilot is separate.
    if torch.cuda.is_available():
        torch.manual_seed(0)
        with torch.inference_mode():
            layer=bitsandbytes.nn.Linear4bit(64,64,bias=False,
                compute_dtype=torch.float16,quant_type='nf4').to('cuda:0')
            output=layer(torch.randn(1,64,device='cuda:0',dtype=torch.float16))
            torch.cuda.synchronize()
            assert torch.isfinite(output).all(), 'NF4 test produced nonfinite output'
        print('Small CUDA NF4 forward check passed on',torch.cuda.get_device_name(0))
    else:
        print('GPU is unavailable. Enable a Kaggle GPU before model execution.')
    print('Notebook imports passed.')
    print('torch:',torch.__version__,'torchvision:',torchvision.__version__)
    print('transformers:',transformers.__version__,'Pillow:',PIL.__version__)
    """
    probe=textwrap.dedent(probe)
    check=subprocess.run([sys.executable,'-c',probe],capture_output=True,text=True)
    print(check.stdout)
    if check.returncode:
        print(check.stderr)
        raise RuntimeError('Notebook import check failed. Fix the traceback above before proceeding.')

    # Check the dependencies reached from this notebook, including selected extras.
    # A conflict in an unrelated package such as dopamine-rl must not block grading.
    def notebook_dependency_errors(roots):
        pending=[(canonicalize_name(name),frozenset()) for name in roots]
        seen=set(); errors=set()
        while pending:
            name,extras=pending.pop()
            if (name,extras) in seen: continue
            seen.add((name,extras))
            version=installed_version(name)
            if version is None:
                errors.add(f'Missing notebook dependency: {name}')
                continue
            for text in metadata.requires(name) or []:
                requirement=Requirement(text)
                active=(requirement.marker is None or any(
                    requirement.marker.evaluate({'extra':extra}) for extra in set(extras)|{''}))
                if not active: continue
                dependency=canonicalize_name(requirement.name)
                actual=installed_version(dependency)
                if actual is None:
                    errors.add(f'{name} requires {requirement}; {dependency} is missing')
                elif requirement.specifier and not requirement.specifier.contains(actual):
                    errors.add(f'{name} requires {requirement}; installed {dependency}=={actual}')
                pending.append((dependency,frozenset(requirement.extras)))
        return sorted(errors)

    roots=list(TARGET)+['Pillow','torch','torchvision','numpy','pandas','scipy',
                        'scikit-learn','matplotlib','ipython']
    errors=notebook_dependency_errors(roots)
    if errors:
        print('Broken requirements used by this notebook:')
        print('\n'.join(errors))
        raise RuntimeError('A notebook dependency is incompatible. Fix the requirements listed above.')

    report=subprocess.run([sys.executable,'-m','pip','check'],capture_output=True,text=True)
    report_text=report.stdout or report.stderr
    report_folder=Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()
    (report_folder/'knee_oa_environment_dependency_check.txt').write_text(report_text,encoding='utf-8')
    if report.returncode:
        print('Other installed-package conflicts (not reached from notebook dependencies):')
        print(report_text)
        print('These unrelated conflicts are logged; no Gym/Dopamine changes are needed for this notebook.')
    else:
        print('Full environment check: no broken requirements found.')
    print('Notebook dependency and import checks passed. Continue to section 2.')
    print('Actual model inference and adapter training still require the development pilot.')

    return True

SETUP_READY = prepare_notebook_environment()
if SETUP_READY:
    print('NOTEBOOK READY — continue to section 2 with mode=pilot.')
else:
    print('PAUSED FOR KERNEL RESTART — keep this notebook and rerun setup after restarting the Python kernel.')


## 2 Configure the run and define analysis utilities

Edit the `CFG` dictionary near the top of the next cell. This copy starts with `mode='study'` for your current run. Use `mode='pilot'` to repeat the development pilot. The full study uses `mode='study'` and writes to a separate directory automatically. Seed 42, the two CNN learning rates, the two adapter learning rates, 2,000 bootstrap resamples and the bounded robustness/timing cohorts follow the proposal.

The six-hour adapter wall-time cap is an explicit implementation budget covering selection and refitting; adjust it on development data before the study protocol is frozen. It is not a hardware performance guarantee. Full-data adaptation, additional models and additional seeds are not silently added. Optional eight-bit inference is disabled by default.

**Resume:** persist the complete private Kaggle output as a Kaggle Dataset, attach it to a new session, and set `resume_from` to its prior `pilot` or `study` directory. The code copies missing outputs, verifies the run contract, and skips inference rows already recorded. Completed final CNN and adapter checkpoints are reused. An interrupted training fit restarts that fit; optimizer state is not resumed. Keep the same software, hardware, data and configuration for a strict frozen-run resume. Use a new `output_root` for changed experiments.

Metrics use the proposal definitions: five-class macro-F1 includes all five grades; balanced accuracy averages recall over grades present in that evaluated cohort; Brier is the sum of five class errors without dividing by five. QWK is undefined when its expected disagreement is zero and is reported as missing. Undefined bootstrap resamples are counted, not replaced with zero.


In [ ]:
if not globals().get('SETUP_READY', False):
    raise RuntimeError('Setup has not passed yet. Run the first setup cell. If it says RESTART REQUIRED, restart the Python kernel and rerun setup before section 2.')

import os, sys, json, math, random, hashlib, time, gc, re, io, zipfile, shutil, subprocess
from pathlib import Path
from collections import defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
import importlib.metadata as md
import numpy as np
import pandas as pd
from scipy.optimize import minimize_scalar
from scipy.special import softmax, log_softmax
from PIL import Image, ImageFilter
import matplotlib.pyplot as plt
from IPython.display import display, FileLink, Markdown

# Edit this configuration before running the notebook.
CFG = dict(
    mode='study',                         # Current study run; use 'pilot' for a development pilot
    data_root='',                         # folder directly containing train, val, test; blank = discover
    output_root='/kaggle/working/knee_oa_reproduction',
    resume_from='',                        # optional prior run directory mounted as a Kaggle Dataset
    model_id='Qwen/Qwen2.5-VL-3B-Instruct', model_revision='66285546d2b821cf421d4f5eb2576359d3770cd3',
    model_local_path='',                   # optional HF snapshot with processor and model files
    seed=42, max_context=2048, image_size=224, quantization_bits=4,
    cnn_batch=32, cnn_workers=2, cnn_lrs=[1e-4, 3e-4], cnn_epochs=20, cnn_patience=3,
    adapter_enabled=True, adapter_lrs=[1e-4, 2e-4], adapter_epochs=2,
    lora_rank=8, lora_alpha=16, lora_dropout=0.05, accumulation=8,
    adapter_max_wall_hours=6.0,             # explicit compute cap for selection + refit
    bootstrap=2000, timing_repeats=3, timing_warmup=3,
    robustness_groups=100, timing_groups=100,
    optional_8bit=False,
    grouping_verified=False,               # change only with authoritative source evidence
    grouping_evidence='',                  # citation / source and verification date
    source_notes='Folder labels 0–4; source reader/visit conventions remain unresolved.',
    strict_archive_counts=True,
)
# Resume preserves the original contract, including its historical config values.
if CFG['resume_from']:
    resume_origin = Path(CFG['resume_from'])
    resume_contract = resume_origin/'protocol'/'run_contract.json'
    if not resume_contract.is_file():
        raise FileNotFoundError(f'Resume directory must directly contain protocol/: {resume_origin}')
    resume_paths = {k:CFG[k] for k in ['data_root','output_root','resume_from']}
    CFG = {**json.loads(resume_contract.read_text())['config'], **resume_paths}
assert CFG['mode'] in {'pilot','study'}
assert CFG['quantization_bits'] == 4, 'Core protocol is four-bit NF4. Eight-bit is a separate sensitivity.'
if CFG['grouping_verified']:
    assert CFG['grouping_evidence'].strip(), 'Document the authoritative identifier evidence.'
SEED = CFG['seed']
ROOT = Path(CFG['output_root']) / CFG['mode']
ROOT.mkdir(parents=True, exist_ok=True)
if CFG['resume_from']:
    origin = Path(CFG['resume_from'])
    assert origin.resolve() != ROOT.resolve()
    assert origin.is_dir()
    # Copy once, and never overwrite newer local progress.
    for f in origin.rglob('*'):
        dest = ROOT / f.relative_to(origin)
        if f.is_file() and not dest.exists():
            dest.parent.mkdir(parents=True, exist_ok=True); shutil.copy2(f, dest)
for d in ['audit','protocol','predictions','checkpoints','logs','figures','tables']:
    (ROOT/d).mkdir(exist_ok=True)

def clean_json(x):
    if isinstance(x, dict): return {str(k): clean_json(v) for k,v in x.items()}
    if isinstance(x, (list,tuple)): return [clean_json(v) for v in x]
    if isinstance(x, np.ndarray): return clean_json(x.tolist())
    if isinstance(x, np.generic): return clean_json(x.item())
    if isinstance(x, float) and not math.isfinite(x): return None
    if isinstance(x, Path): return str(x)
    return x

def digest(x):
    return hashlib.sha256(json.dumps(clean_json(x),sort_keys=True).encode()).hexdigest()

def dump_json(path, obj):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(clean_json(obj),indent=2,sort_keys=True),encoding='utf-8')
    tmp.replace(path)

def file_hash(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()

def seed_all(s=SEED):
    random.seed(s); np.random.seed(s)
    if 'torch' in globals():
        torch.manual_seed(s); torch.cuda.manual_seed_all(s)
        torch.backends.cudnn.benchmark=False
        torch.backends.cudnn.deterministic=True
        torch.use_deterministic_algorithms(True,warn_only=True)

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
os.environ.setdefault('TOKENIZERS_PARALLELISM','false')
seed_all()
plt.rcParams.update({'font.size':10, 'axes.spines.top':False,'axes.spines.right':False,
                     'figure.dpi':110,'savefig.bbox':'tight','pdf.fonttype':42})
COLORS=['#22577a','#d76a36','#50805f','#8e649e','#ae862d','#5697a1','#bc6388']

def save_table(number, df, title):
    path=ROOT/'tables'/f'Table_{number:02d}.csv'
    df.to_csv(path,index=False)
    display(Markdown(f'**Table {number}. {title}**')); display(df)
    return df

CAPTIONS={}
def save_figure(number,fig,caption):
    fig.tight_layout()
    fig.savefig(ROOT/'figures'/f'Figure_{number:02d}.pdf')
    fig.savefig(ROOT/'figures'/f'Figure_{number:02d}.png',dpi=160)
    CAPTIONS[str(number)]=caption
    dump_json(ROOT/'figures'/'captions.json',CAPTIONS)
    plt.show(); plt.close(fig)
    display(Markdown(f'**Figure {number}.** {caption}'))

def metric_values(y,pred):
    y=np.asarray(y,dtype=int); pred=np.asarray(pred,dtype=int)
    if len(y)==0:
        return dict(QWK=np.nan,macro_F1=np.nan,balanced_accuracy=np.nan,
                    accuracy=np.nan,MAE=np.nan,large_error=np.nan)
    cm=np.bincount(5*y+pred,minlength=25).reshape(5,5).astype(float)
    actual=cm.sum(1); guessed=cm.sum(0); diag=np.diag(cm)
    recall=np.divide(diag,actual,out=np.zeros(5),where=actual>0)
    f1=np.divide(2*diag,actual+guessed,out=np.zeros(5),where=actual+guessed>0)
    w=(np.arange(5)[:,None]-np.arange(5)[None,:])**2/16
    expected=np.outer(actual,guessed)/len(y)
    den=(w*expected).sum()
    return dict(QWK=1-(w*cm).sum()/den if den>0 else np.nan,
                macro_F1=f1.mean(),balanced_accuracy=recall[actual>0].mean(),
                accuracy=float((y==pred).mean()),MAE=float(abs(y-pred).mean()),
                large_error=float((abs(y-pred)>=2).mean()))

def valid_rows(df):
    return df.loc[df.status.eq('ok')].copy()

def logits_array(df): return df[[f'score_{j}' for j in range(5)]].to_numpy(float)

def probabilities(df,T=1.): return softmax(log_softmax(logits_array(df),axis=1)/T,axis=1)

def calibration_metrics(y,p):
    y=np.asarray(y,int)
    if len(y)==0: return dict(NLL=np.nan,Brier=np.nan,ECE=np.nan)
    nll=-np.log(np.clip(p[np.arange(len(y)),y],1e-15,1)).mean()
    brier=((p-np.eye(5)[y])**2).sum(1).mean()
    bins=reliability(y,p,np.linspace(0,1,16))
    ece=(bins.n/len(y)*(bins.confidence-bins.correctness).abs()).sum()
    return dict(NLL=nll,Brier=brier,ECE=ece)

def reliability(y,p,edges):
    conf=p.max(1); correct=p.argmax(1)==np.asarray(y)
    idx=np.clip(np.searchsorted(edges,conf,side='right')-1,0,len(edges)-2)
    rows=[]
    for b in range(len(edges)-1):
        m=idx==b
        rows.append(dict(lower=edges[b],upper=edges[b+1],n=int(m.sum()),
                         confidence=float(conf[m].mean()) if m.any() else np.nan,
                         correctness=float(correct[m].mean()) if m.any() else np.nan))
    return pd.DataFrame(rows)

def fit_temperature(df):
    v=valid_rows(df)
    if not len(v): raise ValueError('No complete calibration predictions; cannot fit temperature.')
    y=v.y.to_numpy(int); z=log_softmax(logits_array(v),axis=1)
    fit=minimize_scalar(lambda t:-log_softmax(z/np.exp(t),axis=1)[np.arange(len(y)),y].mean(),
                        bounds=(-4,4),method='bounded')
    assert fit.success
    T=float(np.exp(fit.x)); p=softmax(z/T,axis=1)
    assert np.array_equal(p.argmax(1),z.argmax(1))
    return dict(T=T,n=len(v),eligible=len(df),at_bound=abs(fit.x)>3.95)

def cluster_indices(groups,B,seed=SEED):
    groups=np.asarray(groups); unique=np.unique(groups)
    if not len(unique): return
    buckets=[np.flatnonzero(groups==g) for g in unique]
    rng=np.random.default_rng(seed)
    for _ in range(B):
        yield np.concatenate([buckets[k] for k in rng.integers(0,len(buckets),len(buckets))])

def interval(values):
    values=np.asarray(values,float); values=values[np.isfinite(values)]
    return (np.quantile(values,.025),np.quantile(values,.975),len(values)) if len(values) else (np.nan,np.nan,0)

def summary_row(name,df,B=None):
    v=valid_rows(df); out=dict(condition=name,eligible=len(df),valid=len(v),
          completion=len(v)/len(df) if len(df) else np.nan,
          all_case_accuracy=float((v.y.to_numpy()==v.pred.to_numpy()).sum()/len(df)) if len(df) else np.nan)
    out.update(metric_values(v.y,v.pred))
    if len(v):
        samples=defaultdict(list)
        for ix in cluster_indices(v.group_id,CFG['bootstrap'] if B is None else B):
            for k,val in metric_values(v.y.to_numpy()[ix],v.pred.to_numpy()[ix]).items(): samples[k].append(val)
        for k,vals in samples.items():
            lo,hi,n=interval(vals); out.update({k+'_lo':lo,k+'_hi':hi,k+'_bootstrap_valid':n})
    return out

def paired_effect(a,b,label,metric='QWK',B=None):
    # Positive = A minus B. Inner join AFTER failure filtering: common-complete estimand.
    d=valid_rows(a).merge(valid_rows(b),on='case_id',suffixes=('_a','_b'),validate='one_to_one')
    if not len(d): return dict(contrast=label,metric=metric,n=0,effect=np.nan,lo=np.nan,hi=np.nan,p=np.nan)
    assert np.array_equal(d.y_a,d.y_b) and np.array_equal(d.group_id_a,d.group_id_b)
    y=d.y_a.to_numpy(); pa=d.pred_a.to_numpy(); pb=d.pred_b.to_numpy()
    est=metric_values(y,pa)[metric]-metric_values(y,pb)[metric]
    samples=[metric_values(y[ix],pa[ix])[metric]-metric_values(y[ix],pb[ix])[metric]
             for ix in cluster_indices(d.group_id_a,CFG['bootstrap'] if B is None else B)]
    lo,hi,n=interval(samples); vals=np.asarray(samples); vals=vals[np.isfinite(vals)]
    # Approximate two-sided centred bootstrap test. Declared secondary families use Holm.
    p=(1+int((abs(vals-est)>=abs(est)).sum()))/(len(vals)+1) if len(vals) and np.isfinite(est) else np.nan
    return dict(contrast=label,metric=metric,n=len(d),excluded_a=len(a)-len(d),excluded_b=len(b)-len(d),
                effect=est,lo=lo,hi=hi,p=p,bootstrap_valid=n)

def holm(df):
    df=df.copy(); df['p_holm']=np.nan
    idx=df.index[df.p.notna()].tolist(); idx=sorted(idx,key=lambda i:df.loc[i,'p'])
    running=0
    for k,i in enumerate(idx):
        running=max(running,min(1,(len(idx)-k)*df.loc[i,'p'])); df.loc[i,'p_holm']=running
    return df

print('Mode:',CFG['mode'],'| Outputs:',ROOT)
print('Inference will use pixels and prompts only; reference grades are evaluator-only fields.')


## 3 Audit images and freeze data allocations

The audit records byte/pixel hashes, image dimensions, grade folders, neutral case keys, and prefix linkage. It rejects archive-count changes, unreadable files, exact duplicates and retained-split prefix overlap. A cross-split dHash screen flags perceptually similar images for review; it does not certify independence or automatically change labels/exclusions.

The fixed allocations contain 206 calibration groups, 207 protocol-development groups, and 578 training groups for the adapter/matched CNN. Both knees remain together. Full-CNN selection and subset-model selection each use their own internal training-group split. S1–S3 contain one labelled training image per grade in fixed grade order, using distinct groups. No validation or test image is an example.

The 50-image pilot is balanced for feasibility and must not be interpreted as prevalence-representative final performance. If the fixed development subset cannot supply ten images of every grade, the notebook stops; it does not borrow calibration or test cases.

**Audit progress.** This cell opens and checks every attached image again in study mode. It prints progress every 250 images, then reports similarity-screening progress every 500 images. Each image file is read once for decoding and its byte fingerprint. Counts, fingerprints, duplicate checks and grouping rules are retained. Let the cell finish before section 4. A `KeyboardInterrupt` means execution was interrupted. The traceback alone does not identify what triggered the interruption.


In [ ]:
def find_data_root():
    if CFG['data_root']:
        p=Path(CFG['data_root'])
        assert all((p/s).is_dir() for s in ['train','val','test'])
        return p
    candidates=sorted({p.parent for p in Path('/kaggle/input').rglob('train')
                       if p.is_dir() and (p.parent/'val').is_dir() and (p.parent/'test').is_dir()})
    if len(candidates)!=1:
        raise ValueError(f'Found {len(candidates)} dataset roots. Set data_root explicitly: {candidates}')
    return candidates[0]

def dhash(im):
    x=np.asarray(im.convert('L').resize((9,8),Image.Resampling.LANCZOS))
    bits=(x[:,1:]>x[:,:-1]).ravel()
    return int.from_bytes(np.packbits(bits).tobytes(),'big')

def audit_archive(root):
    rows=[]; errors=[]
    audit_started=time.perf_counter()
    print('Starting image audit. Progress is printed every 250 images.', flush=True)
    for split in ['train','val','test','auto_test']:
        folder=root/split
        if not folder.is_dir(): continue
        print(f'Listing {split} images...', flush=True)
        paths=[path for path in sorted(folder.rglob('*'))
               if path.is_file() and path.suffix.lower() in {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}]
        print(f'{split}: {len(paths):,} images to inspect.', flush=True)
        for index,path in enumerate(paths,start=1):
            rel=path.relative_to(root).as_posix()
            if (index-1)%250==0:
                print(f'{split}: opening image {index:,}/{len(paths):,} ({path.name})', flush=True)
            try:
                grade=int(path.parent.name); assert grade in range(5)
                match=re.match(r'^(\d{7})(?:\D|$)',path.stem)
                assert match, f'Unrecognised prefix convention: {path.name}'
                prefix=match.group(1)
                image_bytes=path.read_bytes()
                with Image.open(io.BytesIO(image_bytes)) as source:
                    source.load(); im=source.convert('L'); arr=np.asarray(im)
                    width,height=im.size; source_mode=source.mode
                group_id='g_'+hashlib.sha256(prefix.encode()).hexdigest()[:16]
                case_id='c_'+hashlib.sha256(rel.encode()).hexdigest()[:20]
                pixel=hashlib.sha256(str(arr.shape).encode()+arr.tobytes()).hexdigest()
                rows.append(dict(case_id=case_id,group_id=group_id,split=split,y=grade,
                                 relative_path=rel,width=width,height=height,source_mode=source_mode,
                                 byte_hash=hashlib.sha256(image_bytes).hexdigest(),
                                 pixel_hash=pixel,dhash=f'{dhash(im):016x}'))
            except Exception as e: errors.append(dict(path=rel,error=str(e)))
            if index%250==0 or index==len(paths):
                elapsed=time.perf_counter()-audit_started
                print(f'{split}: inspected {index:,}/{len(paths):,}; '
                      f'{len(rows):,} readable images overall; elapsed {elapsed:.1f}s.', flush=True)
    print('Checking image counts, dimensions, duplicates and filename groups...', flush=True)
    dump_json(ROOT/'audit'/'unreadable_or_unmapped.json',errors)
    if errors: raise ValueError(f'{len(errors)} audit errors; inspect audit log before continuing.')
    df=pd.DataFrame(rows); assert len(df)>0
    df.to_csv(ROOT/'audit'/'manifest.csv',index=False)
    assert df.case_id.is_unique
    core=df[df.split.isin(['train','val','test'])].copy()
    overlap=core.groupby('group_id').split.nunique()
    assert overlap.max()==1, 'Group prefix overlaps retained partitions.'
    assert core.groupby(['split','group_id']).size().eq(2).all(), 'Expected two linked knees per core prefix.'
    assert core.width.eq(224).all() and core.height.eq(224).all(), 'Image dimensions differ from approved audit.'
    for key in ['byte_hash','pixel_hash']:
        dup=core[core.duplicated(key,keep=False)]
        dup.to_csv(ROOT/'audit'/f'exact_duplicates_{key}.csv',index=False)
        assert not len(dup), f'Exact {key} duplicates differ from proposal audit; resolve before formal analysis.'
    counts=pd.crosstab(df.split,df.y).reindex(columns=range(5),fill_value=0)
    counts['total']=counts.sum(1); counts.to_csv(ROOT/'audit'/'grade_counts.csv')
    expected={'train':[2286,1046,1516,757,173], 'val':[328,153,212,106,27],
              'test':[639,296,447,223,51], 'auto_test':[604,275,403,200,44]}
    if CFG['strict_archive_counts']:
        for split,vals in expected.items():
            if split in counts.index: assert counts.loc[split,list(range(5))].tolist()==vals, f'{split} count mismatch'
    auto=df[df.split.eq('auto_test')]
    dump_json(ROOT/'audit'/'auto_test_exclusion.json',dict(images=len(auto),groups=auto.group_id.nunique(),
        groups_also_in_test=len(set(auto.group_id)&set(df.loc[df.split.eq('test'),'group_id'])),
        role='Excluded from all fitting, calibration, thresholds and evaluation.'))
    display(counts)
    print(f'Image audit complete: {len(df):,} images in '
          f'{time.perf_counter()-audit_started:.1f}s.', flush=True)
    return df

def similarity_audit(core):
    # Pigeonhole candidate search: Hamming distance <=4 implies a match in at least
    # one of five nonoverlapping bit segments. All candidates then receive exact dHash distance.
    vals=[int(x,16) for x in core.dhash]; buckets=defaultdict(list); found=[]
    splits=core['split'].to_numpy(); case_ids=core['case_id'].to_numpy()
    print(f'Starting cross-split similarity screening for {len(core):,} images...', flush=True)
    offsets=[(0,13),(13,13),(26,13),(39,13),(52,12)]
    for i,v in enumerate(vals):
        candidates=set()
        for segment,(offset,width) in enumerate(offsets):
            candidates.update(buckets[(segment,(v>>offset)&((1<<width)-1))])
        for j in sorted(candidates):
            if splits[i]==splits[j]: continue
            distance=(v^vals[j]).bit_count()
            if distance<=4:
                found.append(dict(case_a=case_ids[j],case_b=case_ids[i],
                                  split_a=splits[j],split_b=splits[i],dhash_distance=distance))
        for segment,(offset,width) in enumerate(offsets):
            buckets[(segment,(v>>offset)&((1<<width)-1))].append(i)
        if (i+1)%500==0 or i+1==len(vals):
            print(f'Similarity screening: {i+1:,}/{len(vals):,} images checked.', flush=True)
    pd.DataFrame(found,columns=['case_a','case_b','split_a','split_b','dhash_distance']).to_csv(
        ROOT/'audit'/'cross_split_similarity_flags.csv',index=False)
    print('Cross-split dHash flags:',len(found),'(screening flags; no automatic label changes or exclusions).')

def split_groups(frame,fraction,seed):
    from sklearn.model_selection import train_test_split
    groups=frame.groupby('group_id',sort=True).y.max()
    # Group maximum grade is the declared stratifier, never the model input.
    a,b=train_test_split(groups.index.to_numpy(),test_size=fraction,random_state=seed,stratify=groups.to_numpy())
    return frame[frame.group_id.isin(a)].copy(),frame[frame.group_id.isin(b)].copy()

DATA_ROOT=find_data_root()
MANIFEST=audit_archive(DATA_ROOT)
CORE=MANIFEST[MANIFEST.split.ne('auto_test')].reset_index(drop=True)
similarity_audit(CORE)
MANIFEST_HASH=digest(CORE.drop(columns=['relative_path']).to_dict('records'))
TRAIN=CORE[CORE.split.eq('train')].copy()
VAL=CORE[CORE.split.eq('val')].copy()
TEST=CORE[CORE.split.eq('test')].copy()
# Exactly 206 of the 413 validation groups, sampled without using outputs.
vrng=np.random.default_rng(SEED)
vkeys=np.array(sorted(VAL.group_id.unique())); vrng.shuffle(vkeys)
cal_keys=set(vkeys[:206])
CAL=VAL[VAL.group_id.isin(cal_keys)].copy()
DEV=VAL[~VAL.group_id.isin(cal_keys)].copy()
assert CAL.group_id.nunique()==206 and DEV.group_id.nunique()==207
_,SUBSET=split_groups(TRAIN,578/TRAIN.group_id.nunique(),SEED)
assert SUBSET.group_id.nunique()==578 and set(SUBSET.y)==set(range(5))
FULL_FIT,FULL_SELECT=split_groups(TRAIN,.2,SEED+1)
SUB_FIT,SUB_SELECT=split_groups(SUBSET,.2,SEED+1)

SUPPORTS={}; used=set()
for s in range(1,4):
    chosen=[]
    for grade in range(5):
        pool=TRAIN[(TRAIN.y==grade)&~TRAIN.group_id.isin(used)].sort_values('case_id')
        assert len(pool)>0
        row=pool.sample(1,random_state=SEED+100*s+grade).iloc[0]
        chosen.append(row.to_dict()); used.add(row.group_id)
    SUPPORTS[f'S{s}']=pd.DataFrame(chosen).sort_values('y')
    SUPPORTS[f'S{s}'].to_csv(ROOT/'protocol'/f'support_S{s}.csv',index=False)

assert all((DEV.y==k).sum()>=10 for k in range(5)), \
    'The fixed development allocation lacks 10 cases of a grade. Revise the development-only split prospectively in a new run; do not borrow calibration/test cases.'
PILOT=pd.concat([DEV[DEV.y==k].sort_values('case_id').sample(10,random_state=SEED+k) for k in range(5)])
assert len(PILOT)==50
def index_cases(frame,n,seed):
    keys=np.array(sorted(frame.group_id.unique()))
    chosen=np.random.default_rng(seed).choice(keys,min(n,len(keys)),replace=False)
    return frame[frame.group_id.isin(chosen)].sort_values('case_id').groupby('group_id',sort=True).head(1)

ROBUST=index_cases(TEST,CFG['robustness_groups'],SEED+20)
TIMING=index_cases(TEST,CFG['timing_groups'],SEED+21)
SHUFFLED_IDS=np.roll(ROBUST.case_id.to_numpy(),1)
assert all(ROBUST.case_id.to_numpy()!=SHUFFLED_IDS)
for name,frame in [('calibration',CAL),('protocol_development',DEV),('adapter_subset',SUBSET),
                   ('full_fit',FULL_FIT),('full_select',FULL_SELECT),('subset_fit',SUB_FIT),
                   ('subset_select',SUB_SELECT),('pilot',PILOT),('robustness',ROBUST),('timing',TIMING)]:
    frame.to_csv(ROOT/'protocol'/f'{name}_manifest.csv',index=False)
dump_json(ROOT/'protocol'/'shuffled_controls.json',dict(zip(ROBUST.case_id,SHUFFLED_IDS)))
RUN_CONTRACT=digest(dict(config={k:v for k,v in CFG.items() if k not in ['data_root','output_root','resume_from']},
                         manifest=MANIFEST_HASH))
contract_path=ROOT/'protocol'/'run_contract.json'
if contract_path.exists():
    assert json.loads(contract_path.read_text())['hash']==RUN_CONTRACT, \
        'Changed configuration/data in an existing run. Use a new output_root; never mix predictions.'
dump_json(contract_path,dict(hash=RUN_CONTRACT,manifest_hash=MANIFEST_HASH,config=CFG))
print('Audit fingerprint:',MANIFEST_HASH)
print('Calibration / development images:',len(CAL),len(DEV))
print('20% groups / images:',SUBSET.group_id.nunique(),len(SUBSET))
print('Pilot grade counts:',PILOT.y.value_counts().sort_index().to_dict())

CASE_LOOKUP=CORE.set_index('case_id')
def open_case(case_id):
    # File path stays inside the loader and is never inserted into a prompt.
    with Image.open(DATA_ROOT/CASE_LOOKUP.loc[case_id,'relative_path']) as im:
        return im.convert('RGB').resize((224,224),Image.Resampling.BICUBIC)

def transformed_image(case_id,kind='original'):
    im=open_case(case_id)
    if kind=='original': return im
    if kind=='blank': return Image.new('RGB',im.size,(0,0,0))
    if kind=='shuffled':
        j=dict(zip(ROBUST.case_id,SHUFFLED_IDS))[case_id]; return open_case(j)
    if kind.startswith('blur_'): return im.filter(ImageFilter.GaussianBlur(float(kind.split('_')[1])))
    if kind.startswith('noise_'):
        sigma=float(kind.split('_')[1])
        rng=np.random.default_rng(int(hashlib.sha256(f'{SEED}:{case_id}:noise'.encode()).hexdigest()[:8],16))
        a=np.asarray(im.convert('L'),dtype=float)/255
        a=np.clip(a+rng.normal(0,sigma,a.shape),0,1)
        return Image.fromarray(np.rint(a*255).astype('uint8')).convert('RGB')
    if kind.startswith('jpeg_'):
        out=io.BytesIO(); im.save(out,format='JPEG',quality=int(kind.split('_')[1])); out.seek(0)
        with Image.open(out) as compressed: return compressed.convert('RGB')
    raise ValueError(kind)


## 4 Define the supervised and multimodal models

ResNet-18 uses ImageNet initialization, unweighted cross-entropy, AdamW and deterministic ImageNet normalization; no training augmentation is added in this implementation. CNN hyperparameter selection uses internal-group QWK with patience three, then refits from the same initialization for the selected epoch count on the designated training pool.

The VLM uses its own processor and a fixed 224 × 224 pixel budget. Four-bit NF4 weights use float16 compute. All five grade strings share the same assistant prefix. If all five are single-token continuations, their next-token log probabilities are used. Otherwise the full grade-string log probabilities are summed. The common termination convention is **grade digits only, EOS excluded for every candidate**, with no length normalization. The exact tokenization is checked and exported. The lowest grade wins exact ties.

LoRA targets only language attention query/value projections at rank 8, alpha 16 and dropout 0.05. Base, vision and merger weights remain frozen. Supervised loss covers only the reference grade continuation; no generated explanations are trained. Models are loaded sequentially on one GPU. The model revision is resolved to a commit before experiment runs and retained with the processor settings.


In [ ]:
import torch
import torchvision
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from huggingface_hub import HfApi, hf_hub_download

assert torch.cuda.is_available(), 'Select a Kaggle GPU accelerator, then restart and run again.'
DEVICE=torch.device('cuda:0')
seed_all()
versions={x:md.version(x) for x in ['torch','torchvision','transformers','peft','bitsandbytes',
                                   'accelerate','numpy','pandas','scipy','scikit-learn','matplotlib','pillow']}
# Preserve the frozen environment when resuming an existing study.
env_dir = 'logs' if (ROOT/'protocol'/'frozen_protocol.json').exists() else 'protocol'
env_prefix = 'resume_' if env_dir == 'logs' else ''
dump_json(ROOT/env_dir/(env_prefix+'environment.json'),dict(packages=versions,python=sys.version,
    gpu=torch.cuda.get_device_name(0),cuda=torch.version.cuda,cudnn=torch.backends.cudnn.version(),
    device_memory_GiB=torch.cuda.get_device_properties(0).total_memory/2**30))
(ROOT/env_dir/(env_prefix+'requirements_observed.txt')).write_text(subprocess.check_output(
    [sys.executable,'-m','pip','freeze'],text=True),encoding='utf-8')
print(versions); print('GPU:',torch.cuda.get_device_name(0))

def sync(): torch.cuda.synchronize(DEVICE)
def unload(): gc.collect(); torch.cuda.empty_cache()
def is_oom(e): return isinstance(e,torch.cuda.OutOfMemoryError) or 'out of memory' in str(e).lower()

class ComponentTimer:
    def __init__(self,module):
        self.elapsed=0.; self.start=0.
        self.handles=[module.register_forward_pre_hook(self.before),module.register_forward_hook(self.after)]
    def before(self,*args): sync(); self.start=time.perf_counter()
    def after(self,*args): sync(); self.elapsed+=time.perf_counter()-self.start
    def reset(self): self.elapsed=0.
    def close(self):
        for h in self.handles: h.remove()

CNN_TRANSFORM=transforms.Compose([transforms.ToTensor(),transforms.Normalize(
    mean=[.485,.456,.406],std=[.229,.224,.225])])

class KneeDataset(Dataset):
    def __init__(self,frame): self.frame=frame.reset_index(drop=True)
    def __len__(self): return len(self.frame)
    def __getitem__(self,i):
        row=self.frame.iloc[i]
        return CNN_TRANSFORM(open_case(row.case_id)),int(row.y)

def loader(frame,shuffle=False):
    return DataLoader(KneeDataset(frame),batch_size=CFG['cnn_batch'],shuffle=shuffle,
                      num_workers=CFG['cnn_workers'],pin_memory=True,
                      generator=torch.Generator().manual_seed(SEED))

def new_cnn():
    seed_all()
    model=models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc=torch.nn.Linear(model.fc.in_features,5)
    return model.to(DEVICE)

@torch.inference_mode()
def cnn_validate(model,frame):
    model.eval(); ys=[]; ps=[]
    for x,y in loader(frame):
        ps.extend(model(x.to(DEVICE)).argmax(1).cpu().tolist()); ys.extend(y.tolist())
    return metric_values(ys,ps)

TRAIN_LOG=[]
def cnn_train_run(fit,select,lr,epochs,run_id,refit=False):
    model=new_cnn(); optim=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=1e-4)
    loss_fn=torch.nn.CrossEntropyLoss(); best=-np.inf; best_epoch=1; stale=0; history=[]; steps=0
    sync(); t=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    for epoch in range(1,epochs+1):
        model.train(); total=0.; n=0
        for x,y in loader(fit,True):
            x=x.to(DEVICE); y=y.to(DEVICE); optim.zero_grad(set_to_none=True)
            loss=loss_fn(model(x),y)
            if not torch.isfinite(loss): raise ValueError('Nonfinite CNN loss')
            loss.backward(); optim.step(); steps+=1
            total+=loss.item()*len(y); n+=len(y)
        row=dict(run=run_id,epoch=epoch,lr=lr,loss=total/n)
        if not refit:
            report=cnn_validate(model,select); row.update(report); q=report['QWK']
            if np.isfinite(q) and q>best:
                best=q; best_epoch=epoch; stale=0
            else: stale+=1
        history.append(row); print(row)
        pd.DataFrame(history).to_csv(ROOT/'logs'/f'{run_id}_history.csv',index=False)
        if not refit and stale>=CFG['cnn_patience']: break
    sync(); elapsed=time.perf_counter()-t
    resource=dict(condition=run_id,phase='refit' if refit else 'selection',status='ok',lr=lr,
        epochs=epoch,optimizer_steps=steps,seconds=elapsed,GPU_hours=elapsed/3600,
        peak_GiB=torch.cuda.max_memory_allocated()/2**30,groups=fit.group_id.nunique(),images=len(fit),
        trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad))
    append_training(resource)
    return model,dict(lr=lr,best_QWK=best,best_epoch=best_epoch),history

def append_training(record):
    p=ROOT/'logs'/'training_resources.jsonl'
    with p.open('a') as f: f.write(json.dumps(clean_json(record))+'\n')

def fit_cnn(name,all_train,fit,select):
    target=ROOT/'checkpoints'/f'{name}.pt'
    meta=ROOT/'checkpoints'/f'{name}.json'
    if target.exists() and meta.exists():
        saved=json.loads(meta.read_text()); assert saved['run_contract']==RUN_CONTRACT
        assert saved['checkpoint_hash']==file_hash(target)
        return saved
    results=[]
    for lr in CFG['cnn_lrs']:
        model,result,_=cnn_train_run(fit,select,lr,CFG['cnn_epochs'],f'{name}_lr{lr:g}')
        results.append(result); del model; unload()
    candidates=[r for r in results if np.isfinite(r['best_QWK'])]
    assert candidates, 'No finite selection QWK; resolve the fitting failure.'
    selected=max(candidates,key=lambda r:r['best_QWK'])
    model,_,_=cnn_train_run(all_train,None,selected['lr'],selected['best_epoch'],name+'_refit',True)
    torch.save({k:v.detach().cpu() for k,v in model.state_dict().items()},target)
    del model; unload()
    saved=dict(run_contract=RUN_CONTRACT,selection=results,selected=selected,
               groups=all_train.group_id.nunique(),images=len(all_train),checkpoint_hash=file_hash(target))
    dump_json(meta,saved); return saved

class CNNRunner:
    def __init__(self,name):
        self.name=name; sync(); start=time.perf_counter()
        self.model=models.resnet18(weights=None); self.model.fc=torch.nn.Linear(self.model.fc.in_features,5)
        state=torch.load(ROOT/'checkpoints'/f'{name}.pt',map_location='cpu',weights_only=True)
        self.model.load_state_dict(state); self.model.to(DEVICE).eval(); sync()
        self.load_seconds=time.perf_counter()-start
        self.features=torch.nn.Sequential(*list(self.model.children())[:-1])
        self.timer=ComponentTimer(self.features)
    @torch.inference_mode()
    def predict(self,image,**kwargs):
        self.timer.reset(); sync(); start=time.perf_counter()
        x=CNN_TRANSFORM(image).unsqueeze(0).to(DEVICE); sync(); prep=time.perf_counter()-start
        before=time.perf_counter(); z=self.features(x).flatten(1); sync(); image_s=time.perf_counter()-before
        before=time.perf_counter(); scores=self.model.fc(z)[0].float().cpu().numpy(); sync()
        scoring_s=time.perf_counter()-before; total=time.perf_counter()-start
        return dict(scores=scores,preprocess_s=prep,image_s=image_s,language_s=0.,scoring_s=scoring_s,
                    total_s=total,context_tokens=0,visual_tokens=0,scoring_mode='softmax_head')
    def close(self):
        self.timer.close(); del self.features; del self.model; unload()

P0='Assess the current radiographic knee osteoarthritis severity in this knee X-ray using the Kellgren–Lawrence scale. Choose one grade from 0, 1, 2, 3 or 4. Answer with the grade only.'
RUBRIC=('Kellgren–Lawrence rubric: grade 0, no radiographic osteoarthritis; grade 1, doubtful joint-space narrowing with possible osteophytic lipping; grade 2, definite osteophytes with possible joint-space narrowing; grade 3, definite narrowing with multiple osteophytes, some sclerosis and possible bony deformity; grade 4, marked narrowing with large osteophytes, severe sclerosis and definite bony deformity. ')
PROMPTS=dict(P0=P0,P1=RUBRIC+P0,
    P2='Using the Kellgren–Lawrence scale, assign the current radiographic knee osteoarthritis grade for this X-ray. Select 0, 1, 2, 3 or 4. Return only the grade.',
    P3='Grade the knee osteoarthritis visible on this radiograph according to Kellgren–Lawrence. Use one of 0, 1, 2, 3 or 4 and output that grade only.',
    P4='For the knee X-ray shown, determine its current Kellgren–Lawrence osteoarthritis grade. Your answer must be only 0, 1, 2, 3 or 4.',
    P5='Which Kellgren–Lawrence grade best describes the current knee osteoarthritis severity in this radiograph? Choose 0, 1, 2, 3 or 4. Give the grade alone.')
CONDITIONS={'cnn_full':dict(kind='cnn'), 'cnn_20':dict(kind='cnn'),
            'vlm_p0':dict(kind='vlm',prompt='P0',support=None),
            'vlm_p1':dict(kind='vlm',prompt='P1',support=None),
            **{f'vlm_s{i}':dict(kind='vlm',prompt='P1',support=f'S{i}') for i in range(1,4)},
            'lora_20':dict(kind='adapter',prompt='P0',support=None)}
dump_json(ROOT/'protocol'/'prompts.json',PROMPTS)
dump_json(ROOT/'protocol'/'conditions.json',CONDITIONS)

def model_source():
    path=ROOT/'protocol'/'model_revision.json'
    if path.exists():
        info=json.loads(path.read_text()); return info['source'],info['revision']
    if CFG['model_local_path']:
        source=CFG['model_local_path']; files=sorted(Path(source).glob('*'))
        assert (Path(source)/'config.json').exists()
        weights=[f for f in files if f.suffix=='.safetensors']
        assert weights, 'Local snapshot needs model .safetensors files.'
        hashes={f.name:file_hash(f) for f in files if f.is_file()}
        revision=None
        dump_json(path,dict(source=source,revision=revision,local_hashes=hashes,local_digest=digest(hashes)))
        for name in ['LICENSE','README.md']:
            if (Path(source)/name).exists(): shutil.copy2(Path(source)/name,ROOT/'protocol'/('model_'+name))
    else:
        source=CFG['model_id']; revision=HfApi().model_info(source,revision=CFG['model_revision']).sha
        assert re.fullmatch('[0-9a-f]{40}',revision)
        dump_json(path,dict(source=source,revision=revision))
        for name in ['LICENSE','README.md']:
            f=hf_hub_download(source,name,revision=revision)
            shutil.copy2(f,ROOT/'protocol'/('model_'+name))
    return source,revision

class VLMRunner:
    def __init__(self,adapter=None,bits=4,training=False):
        self.source,self.revision=model_source(); sync(); start=time.perf_counter()
        self.processor=AutoProcessor.from_pretrained(self.source,revision=self.revision,
            min_pixels=224*224,max_pixels=224*224,trust_remote_code=False)
        processor_record=dict(image_processor=self.processor.image_processor.to_dict(),
            tokenizer_class=type(self.processor.tokenizer).__name__,
            chat_template=self.processor.chat_template,source=self.source,revision=self.revision)
        processor_path=ROOT/'protocol'/'processor.json'
        if processor_path.exists():
            assert json.loads(processor_path.read_text())==clean_json(processor_record)
        else: dump_json(processor_path,processor_record)
        quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
            bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16) if bits==4 else BitsAndBytesConfig(load_in_8bit=True)
        self.model=Qwen2_5_VLForConditionalGeneration.from_pretrained(self.source,revision=self.revision,
            quantization_config=quant,torch_dtype=torch.float16,device_map={'':0},
            attn_implementation='sdpa',trust_remote_code=False)
        if training:
            self.model=prepare_model_for_kbit_training(self.model,use_gradient_checkpointing=True,
                gradient_checkpointing_kwargs={'use_reentrant':False})
            targets=[n for n,_ in self.model.named_modules()
                     if re.search(r'(?:^|\.)model\.layers\.\d+\.self_attn\.(q_proj|v_proj)$',n)]
            assert targets, 'No language q/v projections found; inspect architecture before changing protocol.'
            self.model=get_peft_model(self.model,LoraConfig(r=CFG['lora_rank'],lora_alpha=CFG['lora_alpha'],
                lora_dropout=CFG['lora_dropout'],target_modules=targets,bias='none',task_type='CAUSAL_LM'))
            trainable=[n for n,p in self.model.named_parameters() if p.requires_grad]
            assert trainable and all('lora_' in n and 'visual' not in n and 'merger' not in n for n in trainable)
            dump_json(ROOT/'protocol'/'adapter_modules.json',dict(targets=targets,trainable_names=trainable,
                trainable_parameters=sum(p.numel() for p in self.model.parameters() if p.requires_grad)))
            self.model.config.use_cache=False
        elif adapter:
            self.model=PeftModel.from_pretrained(self.model,str(adapter),is_trainable=False)
        self.model.eval()
        visual=[m for n,m in self.model.named_modules() if n=='visual' or n.endswith('.visual')]
        assert len(visual)==1, 'Cannot locate unique visual encoder for timing.'
        self.timer=ComponentTimer(visual[0]); sync(); self.load_seconds=time.perf_counter()-start
        self.bits=bits

    def prepare(self,image,prompt='P0',support=None):
        messages=[]; images=[]
        if support:
            for row in SUPPORTS[support].itertuples():
                messages.append(dict(role='user',content=[{'type':'image'},{'type':'text','text':PROMPTS[prompt]}]))
                messages.append(dict(role='assistant',content=str(row.y)))
                images.append(open_case(row.case_id))
        messages.append(dict(role='user',content=[{'type':'image'},{'type':'text','text':PROMPTS[prompt]}]))
        images.append(image)
        text=self.processor.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
        inputs=self.processor(text=[text],images=images,padding=False,return_tensors='pt')
        length=inputs['input_ids'].shape[1]
        assert length+8<=CFG['max_context'], f'Context {length} exceeds fixed budget; do not truncate images silently.'
        tokenizer=self.processor.tokenizer
        prefix=tokenizer.encode(text,add_special_tokens=False)
        candidates=[]
        for grade in range(5):
            encoded=tokenizer.encode(text+str(grade),add_special_tokens=False)
            assert encoded[:len(prefix)]==prefix, 'Answer changes prefix tokenisation; revise only on development data.'
            suffix=encoded[len(prefix):]
            assert suffix and tokenizer.decode(suffix)==str(grade)
            candidates.append(suffix)
        token_audit=dict(candidate_tokens=candidates,termination='Grade digits only; EOS excluded for all candidates.',
                         normalisation='Sum token log probabilities; no length normalisation.',
                         mode='next_token' if all(len(x)==1 for x in candidates) else 'complete_sequence')
        p=ROOT/'protocol'/'answer_tokenization.json'
        if p.exists(): assert json.loads(p.read_text())==token_audit
        else: dump_json(p,token_audit)
        inputs={k:v.to(DEVICE) for k,v in inputs.items()}
        return inputs,candidates,length

    @torch.inference_mode()
    def predict(self,image,prompt='P0',support=None):
        self.model.eval(); self.timer.reset(); sync(); start=time.perf_counter()
        inputs,candidates,length=self.prepare(image,prompt,support); sync(); prep=time.perf_counter()-start
        forward_start=time.perf_counter()
        if all(len(x)==1 for x in candidates):
            out=self.model(**inputs,use_cache=False)
            sync(); forward_s=time.perf_counter()-forward_start; score_start=time.perf_counter()
            # Log probability over vocabulary; normalising these five scores recovers candidate softmax.
            z=torch.log_softmax(out.logits[0,-1].float(),dim=-1)
            scores=z[[x[0] for x in candidates]].cpu().numpy(); del out
            scoring_mode='next_token'
        else:
            values=[]; score_accum=0.
            for ids in candidates:
                ids=torch.tensor([ids],device=DEVICE)
                batch=dict(inputs); batch['input_ids']=torch.cat([inputs['input_ids'],ids],dim=1)
                batch['attention_mask']=torch.ones_like(batch['input_ids'])
                out=self.model(**batch,use_cache=False); sync(); ts=time.perf_counter()
                z=torch.log_softmax(out.logits[0,length-1:length+ids.shape[1]-1].float(),dim=-1)
                values.append(z.gather(1,ids[0,:,None]).sum().item()); del out
                sync(); score_accum+=time.perf_counter()-ts
            scores=np.array(values); sync(); forward_s=time.perf_counter()-forward_start-score_accum
            score_start=time.perf_counter()-score_accum; scoring_mode='complete_sequence'
        sync(); total=time.perf_counter()-start
        image_s=self.timer.elapsed; scoring_s=total-prep-forward_s
        language_s=forward_s-image_s
        assert language_s>=-1e-3
        visual_tokens=int(np.prod(inputs['image_grid_thw'].cpu().numpy(),axis=1).sum()//
                          self.processor.image_processor.merge_size**2)
        return dict(scores=scores,preprocess_s=prep,image_s=image_s,language_s=max(0,language_s),
                    scoring_s=scoring_s,total_s=total,context_tokens=length,visual_tokens=visual_tokens,
                    scoring_mode=scoring_mode)

    def supervised_loss(self,image,grade):
        inputs,candidates,length=self.prepare(image,'P0',None)
        target=torch.tensor([candidates[int(grade)]],device=DEVICE)
        inputs['input_ids']=torch.cat([inputs['input_ids'],target],dim=1)
        inputs['attention_mask']=torch.ones_like(inputs['input_ids'])
        labels=torch.full_like(inputs['input_ids'],-100); labels[:,-target.shape[1]:]=target
        return self.model(**inputs,labels=labels,use_cache=False).loss

    @torch.inference_mode()
    def generate_check(self,image):
        inputs,_,length=self.prepare(image,'P0',None)
        self.model.eval()
        ids=self.model.generate(**inputs,max_new_tokens=8,do_sample=False,use_cache=True)
        text=self.processor.tokenizer.decode(ids[0,length:],skip_special_tokens=True).strip()
        return dict(text=text,parse_ok=bool(re.fullmatch('[0-4]',text)))

    def close(self):
        self.timer.close(); del self.model; del self.processor; unload()


## 5 Define resumable experiments and feasibility gates

Prediction records include raw five-class scores/probabilities, prompt/support identifiers, processing details, time and failure status. Reference labels and provisional linkage are evaluation-only fields. Out-of-memory failures are recorded; after three consecutive OOMs in an inference condition, its remaining cases are explicitly marked unattempted by the memory gate. Unexpected errors stop execution with the failing record retained.

The adapter gate checks finite, nonzero gradients on a training-only example. Selection uses the exact same internal fit/selection groups as the 20% CNN, then refits on the complete designated 20% pool. If the adapter exceeds memory or the declared compute cap, its infeasibility is reported and no adaptation benefit is estimated. Unexpected implementation errors are surfaced for correction.


In [ ]:
def prediction_path(name,split,variant='original'):
    return ROOT/'predictions'/f'{name}__{split}__{variant}.jsonl'

def read_records(path):
    if not path.exists(): return pd.DataFrame()
    rows=[]
    for line in path.read_text().splitlines():
        if not line.strip(): continue
        # A torn last line after a killed runtime is not silently discarded.
        rows.append(json.loads(line))
    df=pd.DataFrame(rows)
    if len(df):
        assert df.case_id.is_unique, f'Duplicate records in {path}'
        assert df.run_contract.eq(RUN_CONTRACT).all(), 'Cache protocol mismatch'
    return df

def run_cases(runner,name,frame,split,variant='original',prompt=None,support=None,persist=True,cache_tag=None):
    path=prediction_path(name,split,cache_tag or variant)
    prior=read_records(path) if persist else pd.DataFrame()
    if len(prior) and 'model_revision' in prior:
        previous=set(prior.model_revision.dropna())
        expected=getattr(runner,'revision',None)
        assert not previous or previous=={expected}, 'Cached predictions use a different model revision.'
    done=set(prior.case_id) if len(prior) else set()
    rows=prior.to_dict('records') if len(prior) else []
    condition=CONDITIONS.get(name,{})
    prompt=prompt or condition.get('prompt','P0')
    if support is None: support=condition.get('support')
    consecutive_oom=0; blocked=False
    for i,row in enumerate(frame.itertuples()):
        if row.case_id in done: continue
        record=dict(case_id=row.case_id,group_id=row.group_id,y=int(row.y),condition=name,
                    split=split,variant=variant,prompt=prompt,support=support,
                    run_contract=RUN_CONTRACT,status='ok',pred=None,
                    model_revision=getattr(runner,'revision',None),
                    precision='float32' if name.startswith('cnn') else f'int{getattr(runner,"bits",4)}-float16')
        record.update({f'score_{j}':None for j in range(5)})
        record.update({f'raw_p_{j}':None for j in range(5)})
        start=time.perf_counter()
        try:
            if blocked: raise MemoryError('Not attempted after three consecutive out-of-memory failures in this condition.')
            image=transformed_image(row.case_id,variant)
            torch.cuda.reset_peak_memory_stats()
            result=runner.predict(image,prompt=prompt,support=support)
            scores=np.asarray(result.pop('scores'),float)
            assert scores.shape==(5,) and np.isfinite(scores).all(), 'Invalid class score vector'
            p=softmax(scores); assert np.isclose(p.sum(),1.) and np.isfinite(p).all()
            record.update(result,pred=int(scores.argmax()),peak_GiB=torch.cuda.max_memory_allocated()/2**30)
            record.update({f'score_{j}':float(scores[j]) for j in range(5)})
            record.update({f'raw_p_{j}':float(p[j]) for j in range(5)})
            consecutive_oom=0
        except Exception as e:
            status='not_attempted_memory_gate' if isinstance(e,MemoryError) else ('oom' if is_oom(e) else 'error')
            record.update(status=status,error_type=type(e).__name__,error=str(e)[:1000],
                          total_s=time.perf_counter()-start,peak_GiB=torch.cuda.max_memory_allocated()/2**30)
            if status=='oom':
                consecutive_oom+=1; unload()
                if consecutive_oom>=3: blocked=True
            elif status=='error':
                # Persist the failing record, then surface unexpected code/data failures immediately.
                if persist:
                    with path.open('a') as f: f.write(json.dumps(clean_json(record))+'\n')
                raise
        rows.append(record)
        if persist:
            with path.open('a') as f: f.write(json.dumps(clean_json(record))+'\n')
        if (i+1)%25==0 or i+1==len(frame):
            print(name,split,variant,f'{i+1}/{len(frame)}',record['status'])
    df=pd.DataFrame(rows)
    assert set(df.case_id)==set(frame.case_id)
    return df.set_index('case_id').loc[frame.case_id].reset_index()

def adapter_validation(runner,frame):
    # Internal training subset only. No calibration, protocol-development or test labels.
    d=run_cases(runner,'lora_20',frame,'internal_selection',persist=False)
    v=valid_rows(d)
    return metric_values(v.y,v.pred)['QWK'] if len(v)==len(frame) else np.nan

def adapter_fit_run(fit,select,lr,epochs,run_id,deadline,refit=False):
    runner=None; start=time.perf_counter(); steps=0; history=[]
    try:
        seed_all(); runner=VLMRunner(training=True); runner.model.train()
        params=[p for p in runner.model.parameters() if p.requires_grad]
        optim=torch.optim.AdamW(params,lr=lr,weight_decay=1e-4)
        scaler=torch.amp.GradScaler('cuda')
        best=-np.inf; best_epoch=1; torch.cuda.reset_peak_memory_stats()
        for epoch in range(1,epochs+1):
            order=fit.sort_values('case_id').sample(frac=1,random_state=SEED+epoch).reset_index(drop=True)
            losses=[]; runner.model.train()
            for begin in range(0,len(order),CFG['accumulation']):
                if time.perf_counter()>deadline: raise TimeoutError('Declared adapter compute cap reached')
                micro=order.iloc[begin:begin+CFG['accumulation']]
                optim.zero_grad(set_to_none=True)
                for row in micro.itertuples():
                    with torch.autocast(device_type='cuda',dtype=torch.float16):
                        loss=runner.supervised_loss(open_case(row.case_id),int(row.y))
                    if not torch.isfinite(loss): raise ValueError('Nonfinite adapter loss')
                    losses.append(float(loss.item()))
                    scaler.scale(loss/len(micro)).backward()
                scaler.unscale_(optim)
                torch.nn.utils.clip_grad_norm_(params,1.)
                scaler.step(optim); scaler.update(); steps+=1
                if steps%25==0: print(run_id,'epoch',epoch,'step',steps,'mean loss',np.mean(losses[-50:]))
            record=dict(run=run_id,epoch=epoch,lr=lr,loss=np.mean(losses))
            if not refit:
                q=adapter_validation(runner,select); record['QWK']=q
                if np.isfinite(q) and q>best: best=q; best_epoch=epoch
            history.append(record); print(record)
            pd.DataFrame(history).to_csv(ROOT/'logs'/f'{run_id}_history.csv',index=False)
        sync(); seconds=time.perf_counter()-start
        append_training(dict(condition=run_id,phase='refit' if refit else 'selection',status='ok',
            epochs=epoch,lr=lr,optimizer_steps=steps,seconds=seconds,GPU_hours=seconds/3600,
            peak_GiB=torch.cuda.max_memory_allocated()/2**30,groups=fit.group_id.nunique(),images=len(fit),
            trainable_parameters=sum(p.numel() for p in params),effective_batch=CFG['accumulation']))
        if refit:
            runner.model.save_pretrained(ROOT/'checkpoints'/'lora_20',safe_serialization=True)
            runner.processor.save_pretrained(ROOT/'checkpoints'/'lora_20')
        return dict(lr=lr,best_QWK=best,best_epoch=best_epoch)
    except Exception as e:
        append_training(dict(condition=run_id,status='oom' if is_oom(e) else 'failed',
            seconds=time.perf_counter()-start,GPU_hours=(time.perf_counter()-start)/3600,
            optimizer_steps=steps,error=str(e),phase='refit' if refit else 'selection'))
        raise
    finally:
        if runner is not None: runner.close()

def gradient_gate():
    gate=ROOT/'logs'/'adapter_gradient_gate.json'
    if gate.exists(): return json.loads(gate.read_text())
    runner=None; start=time.perf_counter()
    try:
        seed_all(); runner=VLMRunner(training=True); runner.model.train()
        row=SUB_FIT.sort_values('case_id').iloc[0]
        torch.cuda.reset_peak_memory_stats()
        with torch.autocast(device_type='cuda',dtype=torch.float16):
            loss=runner.supervised_loss(open_case(row.case_id),int(row.y))
        assert torch.isfinite(loss)
        loss.backward()
        gradients=[p.grad for p in runner.model.parameters() if p.requires_grad and p.grad is not None]
        assert gradients and all(torch.isfinite(g).all() for g in gradients)
        assert any(g.abs().sum()>0 for g in gradients), 'Adapter gradients are all zero'
        result=dict(status='ok',loss=loss.item(),peak_GiB=torch.cuda.max_memory_allocated()/2**30,
                    seconds=time.perf_counter()-start)
    except Exception as e:
        result=dict(status='oom' if is_oom(e) else 'failed',error=str(e),seconds=time.perf_counter()-start)
        dump_json(gate,result)
        if not is_oom(e): raise
    finally:
        if runner is not None: runner.close()
    dump_json(gate,result); return result

def fit_adapter():
    path=ROOT/'checkpoints'/'lora_20_result.json'
    if path.exists():
        saved=json.loads(path.read_text())
        if saved['status']=='ok':
            assert saved['run_contract']==RUN_CONTRACT
            for name,h in saved['adapter_hashes'].items():
                assert file_hash(ROOT/'checkpoints'/'lora_20'/name)==h, 'Adapter checkpoint changed'
        return saved
    if not CFG['adapter_enabled']:
        result=dict(status='disabled',reason='Adapter disabled in prespecified configuration')
        dump_json(path,result); return result
    gate=gradient_gate()
    if gate['status']!='ok':
        result=dict(status='infeasible',gate=gate); dump_json(path,result); return result
    deadline=time.perf_counter()+CFG['adapter_max_wall_hours']*3600
    try:
        results=[adapter_fit_run(SUB_FIT,SUB_SELECT,lr,CFG['adapter_epochs'],
                                f'lora_lr{lr:g}',deadline) for lr in CFG['adapter_lrs']]
        candidates=[r for r in results if np.isfinite(r['best_QWK'])]
        if not candidates: raise RuntimeError('No finite internal-selection adapter QWK')
        selected=max(candidates,key=lambda r:r['best_QWK'])
        adapter_fit_run(SUBSET,None,selected['lr'],selected['best_epoch'],'lora_20_refit',deadline,True)
        result=dict(status='ok',run_contract=RUN_CONTRACT,selection=results,selected=selected,
                    groups=SUBSET.group_id.nunique(),images=len(SUBSET),
                    adapter_hashes={f.name:file_hash(f) for f in (ROOT/'checkpoints'/'lora_20').glob('*') if f.is_file()})
    except Exception as e:
        result=dict(status='infeasible',reason=str(e),error_type=type(e).__name__)
        dump_json(path,result)
        if not is_oom(e) and not isinstance(e,TimeoutError): raise
    dump_json(path,result); return result

def study_development():
    fit_cnn('cnn_full',TRAIN,FULL_FIT,FULL_SELECT)
    fit_cnn('cnn_20',SUBSET,SUB_FIT,SUB_SELECT)
    for name in ['cnn_full','cnn_20']:
        runner=CNNRunner(name)
        try:
            run_cases(runner,name,CAL,'calibration'); run_cases(runner,name,DEV,'development')
        finally: runner.close()
    runner=VLMRunner()
    try:
        for name in ['vlm_p0','vlm_p1','vlm_s1','vlm_s2','vlm_s3']:
            run_cases(runner,name,CAL,'calibration'); run_cases(runner,name,DEV,'development')
    finally: runner.close()
    adaptation=fit_adapter()
    if adaptation['status']=='ok':
        runner=VLMRunner(adapter=ROOT/'checkpoints'/'lora_20')
        try:
            run_cases(runner,'lora_20',CAL,'calibration'); run_cases(runner,'lora_20',DEV,'development')
        finally: runner.close()
    return adaptation

def freeze_protocol(adaptation):
    temperatures={}; thresholds={}; available=[]; omitted={}
    for name in CONDITIONS:
        if name=='lora_20' and adaptation['status']!='ok':
            omitted[name]=adaptation; continue
        cal=read_records(prediction_path(name,'calibration'))
        dev=read_records(prediction_path(name,'development'))
        assert set(cal.case_id)==set(CAL.case_id) and set(dev.case_id)==set(DEV.case_id)
        if not len(valid_rows(cal)) or not len(valid_rows(dev)):
            omitted[name]=dict(status='no_valid_development_scores'); continue
        temperatures[name]=fit_temperature(cal)
        v=valid_rows(dev); confidence=probabilities(v,temperatures[name]['T']).max(1)
        target=math.ceil(.8*len(dev))
        if len(v)<target:
            thresholds[name]=dict(threshold=0.,target_coverage=.8,attainable=False,
                                  dev_coverage=len(v)/len(dev),ties='All confidence ties retained')
        else:
            threshold=float(np.sort(confidence)[::-1][target-1])
            thresholds[name]=dict(threshold=threshold,target_coverage=.8,attainable=True,
                dev_coverage=float((confidence>=threshold).sum()/len(dev)),ties='All confidence ties retained')
        available.append(name)
    dump_json(ROOT/'protocol'/'temperatures.json',temperatures)
    dump_json(ROOT/'protocol'/'review_thresholds.json',thresholds)
    dump_json(ROOT/'protocol'/'development_omissions.json',omitted)
    assert 'vlm_p0' in available and 'cnn_full' in available, \
        'The minimum direct-VLM/CNN comparison failed development feasibility. Inspect failures and revise the development scope before any final-test inference.'
    frozen_files={}
    for folder in ['protocol','checkpoints']:
        for f in sorted((ROOT/folder).rglob('*')):
            if f.is_file() and f.name!='frozen_protocol.json': frozen_files[str(f.relative_to(ROOT))]=file_hash(f)
    frozen=dict(run_contract=RUN_CONTRACT,files=frozen_files,available=available,omitted=omitted,
        primary='QWK(vlm_p0) minus QWK(cnn_full), common-complete cases, paired prefix bootstrap',
        grouping='verified participant' if CFG['grouping_verified'] else 'provisional seven-digit prefix',
        inference_scope='engineering benchmark; participant independence not established' if not CFG['grouping_verified'] else 'source-confirmed grouping',
        secondary_families={'RQ2':['rubric QWK','examples QWK'],
                            'RQ3':['adapter vs frozen QWK','adapter vs matched CNN QWK'],
                            'RQ4':['scaled-minus-raw NLL per configuration'],
                            'RQ5':['confidence-minus-random error at 60%,80%,90% per configuration'],
                            'RQ6':['six perturbations x measured configurations x QWK and large-error changes']},
        notes=['Validation was previously used by the CNN pilot.',
               'Foundation pretraining overlap with this benchmark cannot be excluded.',
               'Perceptual similarity flags are retained for review; no automatic exclusions.'])
    path=ROOT/'protocol'/'frozen_protocol.json'
    if path.exists():
        assert json.loads(path.read_text())==clean_json(frozen), 'Frozen protocol changed; start a separate run.'
    else: dump_json(path,frozen)
    print('Protocol frozen before test inference:',file_hash(path))
    return frozen,temperatures,thresholds

def verify_frozen(frozen):
    assert frozen['run_contract']==RUN_CONTRACT
    for rel,hash_value in frozen['files'].items():
        assert file_hash(ROOT/rel)==hash_value, f'Frozen file changed: {rel}'

def final_test(frozen):
    verify_frozen(frozen)
    for name in ['cnn_full','cnn_20']:
        if name not in frozen['available']: continue
        runner=CNNRunner(name)
        try: run_cases(runner,name,TEST,'test')
        finally: runner.close()
    runner=VLMRunner()
    try:
        for name in ['vlm_p0','vlm_p1','vlm_s1','vlm_s2','vlm_s3']:
            if name in frozen['available']: run_cases(runner,name,TEST,'test')
    finally: runner.close()
    if 'lora_20' in frozen['available']:
        runner=VLMRunner(adapter=ROOT/'checkpoints'/'lora_20')
        try: run_cases(runner,'lora_20',TEST,'test')
        finally: runner.close()
    for name in frozen['available']:
        d=read_records(prediction_path(name,'test')); v=d.status.eq('ok')
        p=probabilities(d[v],TEMPERATURES[name]['T'])
        for j in range(5): d.loc[v,f'calibrated_p_{j}']=p[:,j]
        d['temperature']=TEMPERATURES[name]['T']
        d.to_csv(ROOT/'predictions'/f'{name}__test_scored.csv',index=False)

def run_pilot():
    runner=VLMRunner()
    try:
        d=run_cases(runner,'vlm_p0',PILOT,'pilot')
        projections=[]
        small=PILOT.sort_values('case_id').groupby('y',sort=True).head(1)
        for name in ['vlm_p0','vlm_p1','vlm_s1','vlm_s2','vlm_s3']:
            smoke=d if name=='vlm_p0' else run_cases(runner,name,small,'pilot_smoke')
            complete=valid_rows(smoke)
            projections.append(dict(condition=name,attempted=len(smoke),complete=len(complete),
                mean_s=float(complete.total_s.mean()) if len(complete) else np.nan,
                projected_main_inference_hours=float(complete.total_s.mean()*(len(VAL)+len(TEST))/3600) if len(complete) else np.nan,
                max_context_tokens=int(complete.context_tokens.max()) if len(complete) else None,
                note='Rough cal+development+test projection; excludes training, controls, timing repeats and downloads.'))
        pd.DataFrame(projections).to_csv(ROOT/'logs'/'pilot_configuration_cost_projections.csv',index=False)
        display(pd.DataFrame(projections))
        checks=[]
        for row in PILOT.sort_values('case_id').head(10).itertuples():
            try: checks.append(dict(case_id=row.case_id,status='ok',**runner.generate_check(open_case(row.case_id))))
            except Exception as e:
                checks.append(dict(case_id=row.case_id,status='oom' if is_oom(e) else 'error',error=str(e),parse_ok=False))
                if not is_oom(e): raise
                unload()
        pd.DataFrame(checks).to_csv(ROOT/'logs'/'pilot_unconstrained_generation.csv',index=False)
        v=valid_rows(d)
        resource=dict(images=len(d),valid=len(v),load_seconds=runner.load_seconds,
            measured_mean_s=float(v.total_s.mean()) if len(v) else None,
            projected_P0_test_hours=float(v.total_s.mean()*len(TEST)/3600) if len(v) else None,
            note='Projection for P0 test inference only; excludes other conditions, training and bootstrap.')
        dump_json(ROOT/'logs'/'pilot_feasibility.json',resource); display(pd.DataFrame([resource]))
        display(pd.DataFrame([summary_row('P0 DEVELOPMENT PILOT ONLY',d,B=200)]))
    finally: runner.close()
    if CFG['adapter_enabled']: display(pd.DataFrame([gradient_gate()]))
    print('Pilot complete. This is development feasibility, not final-test evidence.')


ADAPTATION=None; FROZEN=None; TEMPERATURES={}; THRESHOLDS={}

# Restore in-memory state without fitting again when a frozen protocol is present.
if CFG['mode']=='study' and (ROOT/'protocol'/'frozen_protocol.json').exists():
    FROZEN=json.loads((ROOT/'protocol'/'frozen_protocol.json').read_text())
    verify_frozen(FROZEN)
    TEMPERATURES=json.loads((ROOT/'protocol'/'temperatures.json').read_text())
    THRESHOLDS=json.loads((ROOT/'protocol'/'review_thresholds.json').read_text())
    adaptation_file=ROOT/'checkpoints'/'lora_20_result.json'
    ADAPTATION=json.loads(adaptation_file.read_text()) if adaptation_file.exists() else dict(status='unavailable')
    print('Frozen study restored. Skip 5a–5c; continue with 5d or cached analysis.')


### 5a Run the development pilot

In pilot mode, this cell scores 50 images with P0, smoke-tests the rubric and each support set on five development images, checks ten unconstrained generations, estimates main-inference costs from observed timings, and tests adapter gradients. Review completion, token/context counts, memory and parsing logs before committing to the full study. Cost projections exclude training, controls and repeated timing benchmarks.


In [ ]:
if CFG['mode']=='pilot':
    run_pilot()
else:
    print('Study mode: pilot cell skipped; proceed to development fitting.')


### 5b Fit models and collect development scores

In study mode, this cell performs CNN selection/refitting, frozen VLM calibration/development inference for all five prompting conditions, and bounded adapter fitting when feasible. **It does not evaluate the held-out test partition.** This can require multiple Kaggle sessions. Save the complete private output between sessions.


In [ ]:
if CFG['mode']=='study':
    ADAPTATION=study_development()
    display(pd.DataFrame([ADAPTATION]))


### 5c Fit temperatures and freeze the final protocol

Temperatures use calibration groups only. The target 80% review threshold uses protocol-development confidences, without test labels. All confidence ties at a threshold are retained, so actual development/test coverage may exceed the target. The protocol records manifests, prompts, model/processor revisions, checkpoints, thresholds, secondary test families and actual omissions before test inference. Its file hashes are checked before the final evaluations.

The primary estimand is **QWK(P0) minus QWK(full CNN)** on common-complete test cases. Ninety-five-percent intervals use 2,000 paired group resamples. Without verified identities these are prefix-cluster intervals. Secondary tests use approximate two-sided centered-bootstrap p-values, with Holm adjustment within the declared RQ families. They are not exact permutation tests. Support-set variation and visual-input controls are descriptive.


In [ ]:
if CFG['mode']=='study':
    FROZEN,TEMPERATURES,THRESHOLDS=freeze_protocol(ADAPTATION)
    display(pd.DataFrame.from_dict(TEMPERATURES,orient='index'))
    display(pd.DataFrame.from_dict(THRESHOLDS,orient='index'))


### 5d Make final held-out test predictions

This is the first formal test-inference cell. Every available main configuration uses the same 1,656 eligible test images. Raw predictions are cached and calibrated CSVs are exported. Do not retune prompts, support sets or model settings after examining these results; a revised experiment requires a separately documented run.


In [ ]:
if CFG['mode']=='study':
    final_test(FROZEN)


## 6 RQ1 Direct grading performance

Compare frozen P0 with the full CNN. All-case accuracy counts failures as incorrect. QWK and other grade-only metrics use valid predictions; the primary paired contrast and confusion matrices use the common-complete cohort. Per-grade supports, recall intervals and large errors expose performance hidden by average accuracy.


In [ ]:
LABELS={'cnn_full':'CNN full','cnn_20':'CNN 20%','vlm_p0':'Frozen P0','vlm_p1':'Rubric P1',
        'vlm_s1':'Few shot S1','vlm_s2':'Few shot S2','vlm_s3':'Few shot S3','lora_20':'LoRA 20%'}
GROUP_LABEL='participant' if CFG['grouping_verified'] else 'provisional prefix'
RESULTS={}
if CFG['mode']=='study':
    RESULTS={name:read_records(prediction_path(name,'test')) for name in FROZEN['available']}

def common_valid(names,results=None):
    results=RESULTS if results is None else results
    names=[n for n in names if n in results]
    if not names: return {}
    keys=set.intersection(*(set(valid_rows(results[n]).case_id) for n in names))
    return {n:valid_rows(results[n]).set_index('case_id').loc[sorted(keys)].reset_index() for n in names}

def recall_interval(df,grade):
    v=valid_rows(df); y=v.y.to_numpy(int); p=v.pred.to_numpy(int)
    if not len(v): return np.nan,np.nan,np.nan,0
    den=(y==grade).sum(); est=float((p[y==grade]==grade).mean()) if den else np.nan
    vals=[]
    for ix in cluster_indices(v.group_id,CFG['bootstrap']):
        m=y[ix]==grade
        vals.append(float((p[ix][m]==grade).mean()) if m.any() else np.nan)
    lo,hi,n=interval(vals); return est,lo,hi,n

def plot_metric_points(ax,df,metric,x=None):
    x=np.arange(len(df)) if x is None else np.asarray(x)
    for i,(_,r) in enumerate(df.iterrows()):
        y=r.get(metric,np.nan); lo=r.get(metric+'_lo',np.nan); hi=r.get(metric+'_hi',np.nan)
        if np.isfinite(y):
            ax.plot(x[i],y,'o',color=COLORS[i%len(COLORS)])
            if np.isfinite(lo) and np.isfinite(hi): ax.vlines(x[i],lo,hi,color=COLORS[i%len(COLORS)])
    ax.set_xticks(x,df.condition.map(lambda n:LABELS.get(n,n)),rotation=20,ha='right')
    ax.set_ylabel(metric.replace('_',' '))

def rq1():
    names=[n for n in ['cnn_full','vlm_p0'] if n in RESULTS]
    allrows=pd.DataFrame([summary_row(n,RESULTS[n]) for n in names])
    shared=common_valid(names)
    primary=paired_effect(RESULTS['vlm_p0'],RESULTS['cnn_full'],'Frozen P0 minus CNN full') if len(names)==2 else dict(status='primary contrast unavailable')
    dump_json(ROOT/'tables'/'primary_estimand.json',primary); display(pd.DataFrame([primary]))
    save_table(5,allrows,'Overall grading and completion on all eligible test cases; grade metrics use valid cases')
    fig,axes=plt.subplots(1,max(1,len(names)),figsize=(6*max(1,len(names)),4.6),squeeze=False)
    for ax,n in zip(axes[0],names):
        d=shared[n]; cm=np.bincount(5*d.y.to_numpy(int)+d.pred.to_numpy(int),minlength=25).reshape(5,5)
        ax.imshow(cm,cmap='Blues')
        for i in range(5):
            for j in range(5): ax.text(j,i,str(cm[i,j]),ha='center',va='center',color='white' if cm[i,j]>cm.max()/2 else 'black')
        ax.set(xticks=range(5),yticks=range(5),xlabel='Predicted KL grade',ylabel='Reference KL grade',title=f'{LABELS[n]} · common n={len(d)}')
    save_figure(1,fig,'Paired confusion matrices on the common-complete test cohort. Failures are reported separately in Table 5; rows are reference grades.')
    rows=[]
    for n,d in shared.items():
        for g in range(5):
            y=d.y.to_numpy(int); p=d.pred.to_numpy(int); actual=y==g; guessed=p==g
            rec,lo,hi,bn=recall_interval(d,g)
            precision=float((y[guessed]==g).mean()) if guessed.any() else 0.
            f1=2*precision*rec/(precision+rec) if precision+rec>0 else 0.
            rows.append(dict(condition=n,grade=g,support=int(actual.sum()),recall=rec,recall_lo=lo,recall_hi=hi,
                precision=precision,F1=f1,large_errors=int((abs(y[actual]-p[actual])>=2).sum()),bootstrap_valid=bn))
    t6=save_table(6,pd.DataFrame(rows),'Grade recognition and large ordinal errors on the paired common-complete cohort')
    fig,axes=plt.subplots(1,2,figsize=(12,4.3))
    for i,(n,d) in enumerate(shared.items()):
        error=abs(d.y.to_numpy()-d.pred.to_numpy()).astype(int)
        axes[0].bar(np.arange(5)+(i-.5)*.34,np.bincount(error,minlength=5)/len(d),.34,label=LABELS[n],color=COLORS[i])
        sub=t6[t6.condition.eq(n)]
        axes[1].plot(sub.grade,sub.recall,'o-',label=LABELS[n],color=COLORS[i])
        axes[1].vlines(sub.grade,sub.recall_lo,sub.recall_hi,color=COLORS[i],alpha=.7)
    axes[0].set(xlabel='Absolute KL error',ylabel='Proportion of paired valid cases',xticks=range(5)); axes[0].legend()
    axes[1].set(xlabel='Reference KL grade',ylabel='Recall',ylim=(0,1),xticks=range(5)); axes[1].legend()
    save_figure(2,fig,f'Ordinal error distances and grade-specific recall. Recall bars are 95% {GROUP_LABEL}-bootstrap intervals; rare-grade resamples without support are excluded and counted in Table 6.')

if CFG['mode']=='study': rq1()


## 7 RQ2 Prompts and training-only demonstrations

The fixed contrasts are rubric P1 minus P0, and S1 with examples minus P1 without examples. S2 and S3 are prespecified sensitivity sets. The notebook never selects an example set using test performance.


In [ ]:
def rq2():
    names=[n for n in ['vlm_p0','vlm_p1','vlm_s1'] if n in RESULTS]
    t7=pd.DataFrame([summary_row(n,RESULTS[n]) for n in names]); t7['examples']=[5 if n=='vlm_s1' else 0 for n in names]
    save_table(7,t7,'Fixed prompt and main demonstration comparisons; all-case completion is explicit')
    paired=[]
    for a,b,label in [('vlm_p1','vlm_p0','Rubric minus minimal'),('vlm_s1','vlm_p1','Five examples minus rubric')]:
        if a in RESULTS and b in RESULTS: paired.append(paired_effect(RESULTS[a],RESULTS[b],label))
    if paired:
        p=holm(pd.DataFrame(paired)); p.to_csv(ROOT/'tables'/'RQ2_paired_contrasts.csv',index=False); display(p)
    shared=common_valid(names)
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    plot_metric_points(axes[0],t7,'QWK')
    axes[0].plot(np.arange(len(t7)),t7.macro_F1,'v--',color='#646464',label='Macro-F1')
    axes[0].vlines(np.arange(len(t7)),t7.macro_F1_lo,t7.macro_F1_hi,color='#646464')
    axes[0].plot([],[],'o',color=COLORS[0],label='QWK')
    axes[0].set_ylabel('QWK / macro-F1'); axes[0].legend()
    for i,(n,d) in enumerate(shared.items()):
        rec=[recall_interval(d,g) for g in range(5)]
        axes[1].plot(range(5),[x[0] for x in rec],'o-',color=COLORS[i],label=LABELS[n])
        axes[1].vlines(np.arange(5)+i*.02,[x[1] for x in rec],[x[2] for x in rec],color=COLORS[i],alpha=.5)
    axes[1].set(xlabel='Reference KL grade',ylabel='Recall on shared complete cases',ylim=(0,1),xticks=range(5)); axes[1].legend()
    save_figure(3,fig,f'Prompt and demonstration comparisons. QWK uses each configuration’s complete cases; recall uses their common-complete cohort. Bars are 95% {GROUP_LABEL}-bootstrap intervals.')
    rows=[]
    for n in ['vlm_s1','vlm_s2','vlm_s3']:
        s=SUPPORTS[CONDITIONS[n]['support']]
        if n in RESULTS:
            r=summary_row(n,RESULTS[n])
        else:
            r=dict(condition=n,status='omitted at development feasibility gate',QWK=np.nan,QWK_lo=np.nan,QWK_hi=np.nan,
                   macro_F1=np.nan,macro_F1_lo=np.nan,macro_F1_hi=np.nan)
        r.update(support_set=CONDITIONS[n]['support'],support_images=len(s),support_groups=s.group_id.nunique()); rows.append(r)
    t8=save_table(8,pd.DataFrame(rows),'Prespecified support-set sensitivity; S1 remains the main set')
    fig,axes=plt.subplots(1,2,figsize=(11,4.3))
    for ax,metric in zip(axes,['QWK','macro_F1']):
        plot_metric_points(ax,t8,metric)
        if 'vlm_p1' in RESULTS:
            ref=metric_values(valid_rows(RESULTS['vlm_p1']).y,valid_rows(RESULTS['vlm_p1']).pred)[metric]
            ax.axhline(ref,color='gray',ls='--',label='Rubric, no examples'); ax.legend()
    save_figure(4,fig,'Three fixed training-only support sets evaluated on the same test targets. Intervals quantify case sampling uncertainty; differences between the three sets are descriptive and do not justify selecting the best test set.')

if CFG['mode']=='study': rq2()


## 8 RQ3 Bounded adaptation and matched training budgets

The 20% adapter and CNN share the same downstream label pool and internal selection groups. Foundation pretraining histories differ. If adaptation fails its gate, the displays explicitly say that its effect was not estimated. Resource rows separate hyperparameter selection from final refitting and retain failures.


In [ ]:
def read_training_resources():
    p=ROOT/'logs'/'training_resources.jsonl'
    return pd.DataFrame([json.loads(x) for x in p.read_text().splitlines() if x.strip()]) if p.exists() else pd.DataFrame()

def rq3():
    rows=[]
    budgets={'cnn_full':1.,'cnn_20':SUBSET.group_id.nunique()/TRAIN.group_id.nunique(),'vlm_p0':0.,
             'lora_20':SUBSET.group_id.nunique()/TRAIN.group_id.nunique()}
    for n in budgets:
        if n in RESULTS:
            r=summary_row(n,RESULTS[n]); r['downstream_group_fraction']=budgets[n]; rows.append(r)
    if 'lora_20' not in RESULTS:
        rows.append(dict(condition='lora_20',status=ADAPTATION['status'],reason=ADAPTATION.get('reason','See adaptation gate log'),
                         downstream_group_fraction=budgets['lora_20']))
    t9=save_table(9,pd.DataFrame(rows),'Training-budget comparison; no optional full-data adapter is assumed')
    pairs=[]
    for b in ['vlm_p0','cnn_20']:
        if 'lora_20' in RESULTS and b in RESULTS:
            pairs.append(paired_effect(RESULTS['lora_20'],RESULTS[b],f'LoRA 20% minus {LABELS[b]}'))
    if pairs:
        effects=holm(pd.DataFrame(pairs)); effects.to_csv(ROOT/'tables'/'RQ3_paired_contrasts.csv',index=False); display(effects)
    fig,axes=plt.subplots(1,2,figsize=(11,4.4))
    for ax,metric in zip(axes,['QWK','macro_F1']):
        for i,r in enumerate(rows):
            if metric not in r: continue
            x=r['downstream_group_fraction']; y=r[metric]
            ax.plot(x,y,'o',color=COLORS[i],label=LABELS[r['condition']])
            ax.vlines(x,r.get(metric+'_lo',np.nan),r.get(metric+'_hi',np.nan),color=COLORS[i])
        ax.set(xlabel='Fraction of downstream training groups',ylabel=metric,xticks=[0,.2,1]); ax.legend(fontsize=8)
    save_figure(5,fig,'Measured downstream label budgets and performance, with grouped 95% intervals. Zero denotes no KL gradient fitting, not absence of foundation pretraining. No scaling law or full-data adapter result is inferred.')
    resources=read_training_resources()
    if not resources.empty:
        resources['grouping_unit']=GROUP_LABEL
        resources.to_csv(ROOT/'tables'/'training_resources_all_attempts.csv',index=False)
        t10=resources.copy()
    else: t10=pd.DataFrame([dict(status='No completed training-resource record')])
    if 'lora_20' not in RESULTS:
        t10=pd.concat([t10,pd.DataFrame([dict(condition='lora_20',status=ADAPTATION['status'],error=ADAPTATION.get('reason','gradient gate failed'))])],ignore_index=True)
    save_table(10,t10,'Measured selection and refit resources, including failed attempts; validation-label use is additional')
    fig,axes=plt.subplots(1,2,figsize=(11,4.3))
    if 'lora_20' in RESULTS and 'vlm_p0' in RESULTS:
        d=common_valid(['vlm_p0','lora_20']); a=d['vlm_p0']; b=d['lora_20']
        before=a.y.to_numpy()==a.pred.to_numpy(); after=b.y.to_numpy()==b.pred.to_numpy()
        counts=[int((before&after).sum()),int((~before&after).sum()),int((before&~after).sum()),int((~before&~after).sum())]
        axes[0].bar(['Both correct','Improved','Deteriorated','Both wrong'],counts,color=COLORS[:4])
        axes[0].tick_params(axis='x',rotation=20); axes[0].set_ylabel('Common-complete cases')
        pd.DataFrame(dict(category=['both correct','improved','deteriorated','both wrong'],n=counts)).to_csv(
            ROOT/'tables'/'RQ3_correctness_transitions.csv',index=False)
        for i,(n,s) in enumerate(common_valid(['vlm_p0','lora_20','cnn_20']).items()):
            rec=[recall_interval(s,g) for g in range(5)]
            axes[1].plot(range(5),[x[0] for x in rec],'o-',color=COLORS[i],label=LABELS[n])
            axes[1].vlines(range(5),[x[1] for x in rec],[x[2] for x in rec],color=COLORS[i],alpha=.5)
        axes[1].set(xlabel='Reference KL grade',ylabel='Recall',ylim=(0,1)); axes[1].legend()
    else:
        for ax in axes:
            ax.axis('off'); ax.text(.5,.5,'Adapter effect not estimated\n'+ADAPTATION['status'],ha='center',va='center',transform=ax.transAxes)
    save_figure(6,fig,'Paired correctness transitions and grade recall after adaptation, when feasible. Failed or omitted adaptation is stated explicitly and is not replaced by simulated performance.')

if CFG['mode']=='study': rq3()


## 9 RQ4 Five-candidate calibration

Positive temperature scaling preserves the selected grade. Compare NLL, five-class Brier and 15-bin top-label ECE, with grouped metric intervals and bin counts. The five broader display bands in Table 12 do not replace the 15 ECE bins.


In [ ]:
def calibration_interval(y,p,groups):
    samples=defaultdict(list)
    for ix in cluster_indices(groups,CFG['bootstrap']):
        for key,val in calibration_metrics(y[ix],p[ix]).items(): samples[key].append(val)
    result={}
    for key,vals in samples.items():
        lo,hi,n=interval(vals); result.update({key+'_lo':lo,key+'_hi':hi,key+'_bootstrap_valid':n})
    return result

def paired_scalar_interval(values,groups):
    if not len(values): return dict(effect=np.nan,lo=np.nan,hi=np.nan,p=np.nan,n=0,bootstrap_valid=0)
    values=np.asarray(values,float); est=float(values.mean())
    samples=np.array([values[ix].mean() for ix in cluster_indices(groups,CFG['bootstrap'])])
    lo,hi,bn=interval(samples)
    p=(1+int((abs(samples-est)>=abs(est)).sum()))/(len(samples)+1)
    return dict(effect=est,lo=lo,hi=hi,p=p,n=len(values),bootstrap_valid=bn)

def rq4():
    rows=[]; effects=[]; bin_records=[]
    for name,d in RESULTS.items():
        v=valid_rows(d); y=v.y.to_numpy(int); raw=probabilities(v); T=TEMPERATURES[name]['T']; scaled=probabilities(v,T)
        assert np.array_equal(raw.argmax(1),scaled.argmax(1))
        for state,p,t in [('Raw',raw,1.),('Scaled',scaled,T)]:
            r=dict(condition=name,state=state,T=t,eligible=len(d),valid=len(v),accuracy=float((p.argmax(1)==y).mean()),
                   temperature_at_search_bound=TEMPERATURES[name]['at_bound'])
            r.update(calibration_metrics(y,p)); r.update(calibration_interval(y,p,v.group_id.to_numpy())); rows.append(r)
            bins=reliability(y,p,np.linspace(0,1,16)); bins['condition']=name; bins['state']=state; bin_records.append(bins)
        raw_nll=-np.log(np.clip(raw[np.arange(len(y)),y],1e-15,1))
        cal_nll=-np.log(np.clip(scaled[np.arange(len(y)),y],1e-15,1))
        effects.append(dict(condition=name,contrast='Scaled minus raw NLL',**paired_scalar_interval(cal_nll-raw_nll,v.group_id.to_numpy())))
    save_table(11,pd.DataFrame(rows),'Raw and temperature-scaled five-candidate calibration on complete test predictions')
    effects=holm(pd.DataFrame(effects)); effects.to_csv(ROOT/'tables'/'RQ4_calibration_contrasts.csv',index=False)
    pd.concat(bin_records).to_csv(ROOT/'tables'/'reliability_15_bins_all_conditions.csv',index=False)
    chosen=[n for n in ['vlm_p0','lora_20','cnn_full'] if n in RESULTS][:2]
    fig,axes=plt.subplots(1,len(chosen),figsize=(6*len(chosen),4.7),squeeze=False)
    for ax,name in zip(axes[0],chosen):
        v=valid_rows(RESULTS[name]); y=v.y.to_numpy(int)
        ax.plot([0,1],[0,1],':',color='gray')
        for j,(state,T) in enumerate([('Raw',1.),('Scaled',TEMPERATURES[name]['T'])]):
            b=reliability(y,probabilities(v,T),np.linspace(0,1,16)); b=b[b.n>0]
            ax.plot(b.confidence,b.correctness,'-',color=COLORS[j],alpha=.7)
            ax.scatter(b.confidence,b.correctness,s=18+3*np.sqrt(b.n),color=COLORS[j],label=state)
        ax.set(xlabel='Mean maximum class probability',ylabel='Observed correctness',title=LABELS[name],xlim=(0,1),ylim=(0,1)); ax.legend()
    save_figure(7,fig,'Top-label reliability in 15 fixed equal-width bins; empty bins are omitted, marker area reflects support. Model-level metric intervals appear in Table 11. Scaling does not change the selected grade.')
    name='vlm_p0'; v=valid_rows(RESULTS[name]); y=v.y.to_numpy(int)
    scaled=probabilities(v,TEMPERATURES[name]['T'])
    bins=reliability(y,scaled,np.array([0,.4,.5,.6,.7,1.]))
    save_table(12,bins,'Scaled frozen P0 in five readable display bands; ECE still uses 15 bins')
    fig,axes=plt.subplots(1,2,figsize=(11,4.3))
    for ax,(state,T) in zip(axes,[('Raw',1.),('Scaled',TEMPERATURES[name]['T'])]):
        p=probabilities(v,T); ok=p.argmax(1)==y
        for j,(mask,label) in enumerate([(ok,'Correct'),(~ok,'Incorrect')]):
            ax.hist(p.max(1)[mask],bins=np.linspace(0,1,21),alpha=.55,label=f'{label} n={mask.sum()}',color=COLORS[j])
        ax.set(xlabel='Maximum candidate probability',ylabel='Cases',title=state); ax.legend()
    save_figure(8,fig,'Frozen P0 confidence distributions before and after temperature scaling. Correctness is unchanged; reduced confidence alone is not improved grading.')

if CFG['mode']=='study': rq4()


## 10 RQ5 Selective prediction and review burden

Matched retention ranks each configuration at the same all-eligible coverage, where achievable; ties are broken by neutral case key. A separate frozen threshold is applied unchanged from development. Confidence-versus-random comparisons re-rank inside group resamples. Failures are always sent for review.


In [ ]:
def selection_metrics(y,pred):
    y=np.asarray(y,int); pred=np.asarray(pred,int)
    return dict(n=len(y),errors=int((y!=pred).sum()),large_errors=int((abs(y-pred)>=2).sum()),
                error_risk=float((y!=pred).mean()) if len(y) else np.nan,
                large_risk=float((abs(y-pred)>=2).mean()) if len(y) else np.nan,
                retained_QWK=metric_values(y,pred)['QWK'])

def rank_data(name,d,random_order=False):
    v=valid_rows(d).copy(); v['confidence']=probabilities(v,TEMPERATURES[name]['T']).max(1)
    v['random_key']=[int(hashlib.sha256(f'{SEED}:random:{x}'.encode()).hexdigest()[:12],16) for x in v.case_id]
    return v.sort_values(['random_key','case_id'] if random_order else ['confidence','case_id'],
                         ascending=[True,True] if random_order else [False,True])

def retained_at(name,d,coverage,random_order=False):
    v=rank_data(name,d,random_order); k=math.ceil(coverage*len(d)); attainable=k<=len(v)
    if not attainable:
        return dict(condition=name,policy='random' if random_order else 'confidence',target=coverage,
                    attainable=False,eligible=len(d),valid=len(v),n=0,coverage=np.nan,
                    max_attainable_coverage=len(v)/len(d))
    kept=v.iloc[:k]; r=selection_metrics(kept.y,kept.pred)
    r.update(condition=name,policy='random' if random_order else 'confidence',target=coverage,attainable=True,
             eligible=len(d),valid=len(v),coverage=k/len(d),review_n=len(d)-k)
    return r

def selective_bootstrap(name,d,c):
    # Resample eligible groups, then re-rank each resample; failures remain in coverage denominator.
    samples=defaultdict(list)
    for ix in cluster_indices(d.group_id,CFG['bootstrap']):
        sample=d.iloc[ix].copy()
        a=retained_at(name,sample,c); b=retained_at(name,sample,c,True)
        if not a['attainable'] or not b['attainable']: continue
        samples['error_risk'].append(a['error_risk']); samples['large_risk'].append(a['large_risk'])
        samples['effect'].append(a['error_risk']-b['error_risk'])
    result={}
    for key in ['error_risk','large_risk']:
        lo,hi,bn=interval(samples[key]); result.update({key+'_lo':lo,key+'_hi':hi,key+'_bootstrap_valid':bn})
    est=retained_at(name,d,c)['error_risk']-retained_at(name,d,c,True)['error_risk']
    lo,hi,bn=interval(samples['effect']); values=np.asarray(samples['effect'])
    p=(1+int((abs(values-est)>=abs(est)).sum()))/(len(values)+1)
    effect=dict(condition=name,coverage=c,effect=est,lo=lo,hi=hi,p=p,bootstrap_valid=bn)
    return result,effect

def rq5():
    rows=[]; effects=[]
    for name,d in RESULTS.items():
        for c in [.6,.8,.9]:
            r=retained_at(name,d,c)
            if r['attainable']:
                ci,effect=selective_bootstrap(name,d,c); r.update(ci); effects.append(effect)
            rows.append(r); rows.append(retained_at(name,d,c,True))
    t13=save_table(13,pd.DataFrame(rows),'Matched 60%, 80% and 90% retention, with seeded random references and attainable coverage')
    if effects: holm(pd.DataFrame(effects)).to_csv(ROOT/'tables'/'RQ5_confidence_vs_random.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(12,4.5))
    for i,name in enumerate([n for n in ['cnn_full','vlm_p0','lora_20'] if n in RESULTS]):
        curve=pd.DataFrame([retained_at(name,RESULTS[name],float(c)) for c in np.linspace(.1,1,46)])
        curve.to_csv(ROOT/'tables'/f'risk_coverage_{name}.csv',index=False)
        for ax,key in zip(axes,['error_risk','large_risk']):
            if key not in curve: continue
            ax.plot(curve.coverage,curve.get(key),label=LABELS[name],color=COLORS[i])
            points=t13[(t13.condition==name)&(t13.policy=='confidence')&t13.attainable]
            if len(points): ax.vlines(points.coverage,points[key+'_lo'],points[key+'_hi'],color=COLORS[i])
    if 'vlm_p0' in RESULTS:
        random_curve=pd.DataFrame([retained_at('vlm_p0',RESULTS['vlm_p0'],float(c),True) for c in np.linspace(.1,1,46)])
        for ax,key in zip(axes,['error_risk','large_risk']):
            if key in random_curve: ax.plot(random_curve.coverage,random_curve[key],':',color='gray',label='P0 seeded random')
    for ax,label in zip(axes,['Any-grade-error risk','Large-ordinal-error risk']):
        ax.set(xlabel='Retained fraction of all eligible cases',ylabel=label,xlim=(0,1),ylim=(0,1)); ax.legend(fontsize=8)
    save_figure(9,fig,f'Risk versus all-case coverage. Rankings use calibrated confidence; failures are never released. Bars at 60%, 80% and 90% show 95% {GROUP_LABEL}-bootstrap intervals with ranking recomputed within each resample. Random ordering is a descriptive reference.')
    threshold_rows=[]; accounts=[]
    for name,d in RESULTS.items():
        v=rank_data(name,d); threshold=THRESHOLDS[name]['threshold']; kept=v[v.confidence>=threshold]
        r=selection_metrics(kept.y,kept.pred)
        correct=(v.y.to_numpy()==v.pred.to_numpy()); keep=v.confidence.to_numpy()>=threshold
        accounts.append(dict(condition=name,retained_correct=int((correct&keep).sum()),
            retained_wrong=int((~correct&keep).sum()),referred_correct=int((correct&~keep).sum()),
            referred_wrong=int((~correct&~keep).sum()),failed=len(d)-len(v)))
        r.update(condition=name,threshold=threshold,eligible=len(d),coverage=len(kept)/len(d),
                 development_coverage=THRESHOLDS[name]['dev_coverage'],review_n=len(d)-len(kept))
        vals=defaultdict(list)
        for ix in cluster_indices(d.group_id,CFG['bootstrap']):
            s=d.iloc[ix]; vv=rank_data(name,s); kk=vv[vv.confidence>=threshold]
            vals['coverage'].append(len(kk)/len(s))
            vals['error_risk'].append(selection_metrics(kk.y,kk.pred)['error_risk'])
        for key in vals:
            lo,hi,bn=interval(vals[key]); r.update({key+'_lo':lo,key+'_hi':hi,key+'_bootstrap_valid':bn})
        threshold_rows.append(r)
    save_table(14,pd.DataFrame(threshold_rows),'Unchanged development thresholds at realised test coverage; all ties are retained')
    account=pd.DataFrame(accounts); account.to_csv(ROOT/'tables'/'RQ5_release_review_accounting.csv',index=False)
    fig,ax=plt.subplots(figsize=(12,4.8)); left=np.zeros(len(account))
    for key,color,label in zip(['retained_correct','retained_wrong','referred_correct','referred_wrong','failed'],
        [COLORS[2],COLORS[1],'#b9d5bd','#edc4a9','#9c9c9c'],
        ['Retained correct','Retained wrong','Referred correct','Referred wrong','Inference failed']):
        ax.barh(account.condition.map(LABELS),account[key],left=left,color=color,label=label); left+=account[key]
    ax.set_xlabel('Eligible test cases'); ax.legend(fontsize=8,bbox_to_anchor=(1.01,1),loc='upper left')
    save_figure(10,fig,'Correctness and review accounting under fixed development thresholds. Test coverage may differ from 80%; inference failures always enter review. Thresholds were selected without test labels.')

if CFG['mode']=='study': rq5()


## 11 Run fixed robustness controls and timing measurements

The robustness set contains one fixed index knee from each of 100 distinct test groups, selected before outputs are inspected. Six target-only perturbations use blur σ 0.5/1.0 pixels, grayscale noise σ 0.01/0.03 on the 0–1 scale, and JPEG quality 80/60. Noise is deterministic per case, and support images are unchanged. Five wordings consist of canonical P0 and four fixed paraphrases. Blank and cyclically shuffled target images retain the original case's reference label; these are controls, not additional clinical cases.

The separate timing set also contains one index knee from each of 100 fixed test groups. Warm-up and three repeated, seeded case orders are used for every main configuration with batch one and CUDA synchronization. Target disk reads and model loading are outside warm timing; loading is reported separately. Support images are loaded/processed each attempt without prefix caching. Component means include **processing/transfers as a fourth component**, in addition to visual encoding, language/model overhead and scoring, so they sum honestly to the end-to-end mean. Instrumentation adds synchronization overhead. Reported GPU-hours are allocated-GPU wall hours, not measured utilization.

Optional eight-bit inference uses only the fixed robustness subset for its quality estimate and is labeled separately from full-test quality. It does not replace the four-bit primary result.


In [ ]:
PERTURBATIONS=['blur_0.5','blur_1.0','noise_0.01','noise_0.03','jpeg_80','jpeg_60']

def benchmark_runner(runner,name,prompt='P0',support=None):
    # Load target image before timing; image-to-probabilities includes processing, transfers,
    # encoder, language head and candidate scoring. Disk read of target is excluded.
    path=ROOT/'logs'/f'timing_{name}.jsonl'
    rows=[]
    if path.exists():
        rows=[json.loads(x) for x in path.read_text().splitlines() if x.strip()]
        assert all(r['run_contract']==RUN_CONTRACT for r in rows)
    done={(r['repeat'],r['case_id']) for r in rows}
    first=open_case(TIMING.iloc[0].case_id)
    for _ in range(CFG['timing_warmup']):
        try: runner.predict(first,prompt=prompt,support=support)
        except Exception as e:
            if not is_oom(e): raise
            unload(); break
    for repeat in range(CFG['timing_repeats']):
        order=TIMING.sort_values('case_id').sample(frac=1,random_state=SEED+1000+repeat)
        for i,row in enumerate(order.itertuples()):
            if (repeat,row.case_id) in done: continue
            record=dict(condition=name,repeat=repeat,case_id=row.case_id,group_id=row.group_id,
                status='ok',run_contract=RUN_CONTRACT,load_seconds=runner.load_seconds,
                warmups=CFG['timing_warmup'],batch=1,precision='float32' if name.startswith('cnn') else ('int8-float16' if getattr(runner,'bits',4)==8 else 'NF4-float16'),
                support_cache=False,target_disk_read_included=False)
            image=open_case(row.case_id); sync(); torch.cuda.reset_peak_memory_stats(); start=time.perf_counter()
            try:
                result=runner.predict(image,prompt=prompt,support=support); result.pop('scores')
                record.update(result,peak_GiB=torch.cuda.max_memory_allocated()/2**30)
                assert abs(record['total_s']-sum(record[k] for k in ['preprocess_s','image_s','language_s','scoring_s']))<.01
            except Exception as e:
                record.update(status='oom' if is_oom(e) else 'error',error=str(e),total_s=time.perf_counter()-start)
                if not is_oom(e):
                    with path.open('a') as f: f.write(json.dumps(clean_json(record))+'\n')
                    raise
                unload()
            with path.open('a') as f: f.write(json.dumps(clean_json(record))+'\n')
            rows.append(record)
            if (i+1)%50==0: print('Timing',name,'repeat',repeat+1,f'{i+1}/{len(order)}')
    return pd.DataFrame(rows)

def run_controls():
    verify_frozen(FROZEN)
    for name in ['cnn_full','vlm_p0','lora_20']:
        if name not in FROZEN['available']: continue
        runner=CNNRunner(name) if name.startswith('cnn') else VLMRunner(
            adapter=ROOT/'checkpoints'/'lora_20' if name=='lora_20' else None)
        try:
            for variant in ['original']+PERTURBATIONS:
                run_cases(runner,name,ROBUST,'robustness',variant)
            if name=='vlm_p0':
                for prompt in ['P2','P3','P4','P5']:
                    run_cases(runner,name,ROBUST,'robustness',variant='original',prompt=prompt,cache_tag='wording_'+prompt).to_csv(
                        ROOT/'predictions'/f'wording_{prompt}.csv',index=False)
                for variant in ['blank','shuffled']:
                    run_cases(runner,name,ROBUST,'robustness',variant)
        finally: runner.close()
    for name in FROZEN['available']:
        runner=CNNRunner(name) if name.startswith('cnn') else VLMRunner(
            adapter=ROOT/'checkpoints'/'lora_20' if name=='lora_20' else None)
        try: benchmark_runner(runner,name,**{k:v for k,v in CONDITIONS[name].items() if k in ['prompt','support']})
        finally: runner.close()
    if CFG['optional_8bit']:
        runner=None
        try:
            runner=VLMRunner(bits=8)
            run_cases(runner,'vlm_p0_8bit',ROBUST,'quantization_sensitivity')
            benchmark_runner(runner,'vlm_p0_8bit')
            dump_json(ROOT/'logs'/'optional_8bit.json',dict(status='ok',quality_pool='fixed robustness subset'))
        except Exception as e:
            dump_json(ROOT/'logs'/'optional_8bit.json',dict(status='oom' if is_oom(e) else 'failed',error=str(e)))
            if not is_oom(e): raise
        finally:
            if runner is not None: runner.close()

if CFG['mode']=='study': run_controls()


## 12 RQ6 Robustness and correctness-aware stability

Paired perturbation effects use common-complete pairs and grouped intervals. Stability categories retain incomplete cases and separate stable wrong predictions from stable correct ones. Visual-control accuracy uses every eligible index case as the denominator.


In [ ]:
def rq6():
    robust={}; effects=[]; rows=[]
    for name in ['cnn_full','vlm_p0','lora_20']:
        if name not in RESULTS: continue
        robust[name]={variant:read_records(prediction_path(name,'robustness',variant))
                      for variant in ['original']+PERTURBATIONS}
        for variant,d in robust[name].items():
            r=summary_row(name,d); r['target_condition']=variant; rows.append(r)
            if variant!='original':
                for metric in ['QWK','large_error']:
                    effect=paired_effect(d,robust[name]['original'],f'{name}: {variant} minus original',metric)
                    effect.update(condition=name,target_condition=variant); effects.append(effect)
    t15=save_table(15,pd.DataFrame(rows),'Fixed 100-group perturbation cohort; denominators and grouped intervals are explicit')
    effects=holm(pd.DataFrame(effects)); effects.to_csv(ROOT/'tables'/'RQ6_perturbation_effects.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(13,5.6))
    for ax,metric in zip(axes,['QWK','large_error']):
        for j,name in enumerate(robust):
            sub=effects[(effects.condition==name)&(effects.metric==metric)].set_index('target_condition').reindex(PERTURBATIONS)
            y=np.arange(len(sub))+(j-1)*.16
            ax.plot(sub.effect,y,'o',label=LABELS[name],color=COLORS[j]); ax.hlines(y,sub.lo,sub.hi,color=COLORS[j])
        ax.axvline(0,color='gray',ls=':'); ax.set_yticks(range(len(PERTURBATIONS)),PERTURBATIONS)
        ax.set_xlabel('Change in '+('QWK' if metric=='QWK' else 'large-error risk')); ax.legend(fontsize=8)
    save_figure(11,fig,f'Paired changes from each model’s own unmodified image results. Bars are 95% {GROUP_LABEL}-bootstrap intervals on common-complete pairs. Holm-adjusted tests are supplied separately. Transformations are engineering controls, not validated acquisition shifts.')
    base=robust['vlm_p0']['original']
    wordings=[base.set_index('case_id').reindex(ROBUST.case_id)]
    for prompt in ['P2','P3','P4','P5']:
        d=pd.read_csv(ROOT/'predictions'/f'wording_{prompt}.csv')
        wordings.append(d.set_index('case_id').reindex(ROBUST.case_id))
    complete=np.column_stack([d.status.eq('ok').to_numpy() for d in wordings]).all(1)
    predictions=np.column_stack([pd.to_numeric(d.pred,errors='coerce').to_numpy() for d in wordings])
    stable=complete&(predictions==predictions[:,[0]]).all(1)
    reference=ROBUST.y.to_numpy(int); categories=np.full(len(ROBUST),'Incomplete',dtype=object)
    categories[complete&~stable]='Changed grade'
    categories[stable&(predictions[:,0]==reference)]='Stable correct'
    categories[stable&(predictions[:,0]!=reference)]='Stable wrong'
    category_names=['Stable correct','Stable wrong','Changed grade','Incomplete']
    t16=pd.DataFrame([dict(category=c,n=int((categories==c).sum()),
         percent=100*float((categories==c).mean()),eligible=len(ROBUST)) for c in category_names])
    save_table(16,t16,'Correctness-aware stability across canonical P0 and four fixed paraphrases')
    detail=pd.DataFrame(dict(case_id=ROBUST.case_id,group_id=ROBUST.group_id,y=reference,category=categories))
    detail.to_csv(ROOT/'tables'/'RQ6_case_stability.csv',index=False)
    controls={kind:read_records(prediction_path('vlm_p0','robustness',kind)) for kind in ['original','blank','shuffled']}
    control_summary=pd.DataFrame([summary_row(kind,d) for kind,d in controls.items()])
    control_summary.to_csv(ROOT/'tables'/'RQ6_visual_input_controls.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(12,4.7)); bottom=np.zeros(5)
    grade_counts=[]
    for j,c in enumerate(category_names):
        counts=[int(((categories==c)&(reference==g)).sum()) for g in range(5)]
        axes[0].bar(range(5),counts,bottom=bottom,label=c,color=COLORS[j]); bottom+=counts
        for g,n in enumerate(counts): grade_counts.append(dict(grade=g,category=c,n=n))
    pd.DataFrame(grade_counts).to_csv(ROOT/'tables'/'RQ6_stability_by_grade.csv',index=False)
    axes[0].set(xlabel='Reference KL grade',ylabel='Index knees (one per group)',xticks=range(5)); axes[0].legend(fontsize=8)
    axes[1].bar(control_summary.condition,control_summary.all_case_accuracy,color=COLORS[:3])
    axes[1].set(ylabel='Accuracy over all eligible index cases',ylim=(0,1),xlabel='Visual-input control')
    save_figure(12,fig,'Correctness-aware prompt stability by reference grade and all-case accuracy under target, blank and shuffled-image controls. Stable errors are counted separately. Controls use original references and are not pooled with the main cohort; they do not establish causal explanations.')

if CFG['mode']=='study': rq6()


## 13 RQ7 Measured computational feasibility

Quality and timing use their separately identified cohorts. Mean/p95 latency, GPU memory, loading time, attempts, completion and component means are measured from logs. Timing bootstrap units retain repeated measurements for the same sampled group.


In [ ]:
def rq7():
    rows=[]; components=[]
    quality={n:summary_row(n,d) for n,d in RESULTS.items()}
    for name in list(FROZEN['available'])+(['vlm_p0_8bit'] if CFG['optional_8bit'] else []):
        path=ROOT/'logs'/f'timing_{name}.jsonl'
        if not path.exists(): continue
        d=pd.DataFrame([json.loads(x) for x in path.read_text().splitlines() if x.strip()])
        v=valid_rows(d)
        if name in quality: q=quality[name]; pool='full held-out test, valid predictions'
        else:
            q=summary_row(name,read_records(prediction_path(name,'quantization_sensitivity')))
            pool='100-group robustness subset, eight-bit sensitivity only'
        r=dict(condition=name,QWK=q.get('QWK'),QWK_lo=q.get('QWK_lo'),QWK_hi=q.get('QWK_hi'),
            quality_pool=pool,quality_n=q.get('valid'),attempts=len(d),complete=len(v),
            completion=len(v)/len(d),oom=int(d.status.eq('oom').sum()),
            cases=d.case_id.nunique(),repeats=CFG['timing_repeats'],load_seconds=float(d.load_seconds.iloc[0]),
            precision='float32' if name.startswith('cnn') else ('int8-float16' if name.endswith('8bit') else 'NF4-float16'))
        if len(v):
            r.update(mean_s=float(v.total_s.mean()),p95_s=float(v.total_s.quantile(.95)),peak_GiB=float(v.peak_GiB.max()))
            means=[]; p95=[]
            for ix in cluster_indices(v.group_id,CFG['bootstrap']):
                means.append(v.total_s.to_numpy()[ix].mean()); p95.append(np.quantile(v.total_s.to_numpy()[ix],.95))
            lo,hi,bn=interval(means); r.update(mean_s_lo=lo,mean_s_hi=hi,bootstrap_valid=bn)
            lo,hi,bn=interval(p95); r.update(p95_s_lo=lo,p95_s_hi=hi)
            c=dict(condition=name,valid_measurements=len(v),
                **{k:float(v[k].mean()) for k in ['preprocess_s','image_s','language_s','scoring_s','total_s']})
            assert abs(sum(c[k] for k in ['preprocess_s','image_s','language_s','scoring_s'])-c['total_s'])<.01
            c.update(context_tokens_mean=float(v.context_tokens.mean()),context_tokens_max=int(v.context_tokens.max()),
                     visual_tokens_mean=float(v.visual_tokens.mean()),support_cache=False)
            components.append(c)
        rows.append(r)
    t17=save_table(17,pd.DataFrame(rows),'Warm batch-one inference, loading cost, completion and quality with cohort labels')
    t18=save_table(18,pd.DataFrame(components),'Synchronized component means on the identical successful timed attempts')
    fig,ax=plt.subplots(figsize=(9,5))
    for i,r in enumerate(rows):
        if not np.isfinite(r.get('mean_s',np.nan)) or r.get('QWK') is None: continue
        x=r['mean_s']; y=r['QWK']
        ax.scatter(x,y,s=35+20*r['peak_GiB'],alpha=.8,color=COLORS[i%len(COLORS)],
                   label=LABELS.get(r['condition'],r['condition']))
        ax.hlines(y,r['mean_s_lo'],r['mean_s_hi'],color=COLORS[i%len(COLORS)])
        if r.get('QWK_lo') is not None: ax.vlines(x,r['QWK_lo'],r['QWK_hi'],color=COLORS[i%len(COLORS)])
    ax.set(xscale='log',xlabel='Mean warm latency (seconds, log scale)',ylabel='QWK')
    ax.legend(fontsize=8,bbox_to_anchor=(1.01,1),loc='upper left',markerscale=.6)
    save_figure(13,fig,'Measured quality and runtime. Horizontal and vertical bars are grouped 95% intervals; marker area increases with peak allocated GPU memory. Quality and timing cohorts are separately named in Table 17. No optimal frontier is asserted.')
    fig,ax=plt.subplots(figsize=(12,5)); left=np.zeros(len(t18))
    for key,color,label in zip(['preprocess_s','image_s','language_s','scoring_s'],COLORS,
        ['Processing and transfers','Visual encoder','Language and model overhead','Head / answer scoring']):
        ax.barh(t18.condition.map(lambda n:LABELS.get(n,n)),t18[key],left=left,label=label,color=color); left+=t18[key]
    ax.set_xlabel('Mean seconds per successful attempt'); ax.legend(fontsize=8,bbox_to_anchor=(1.01,1),loc='upper left')
    save_figure(14,fig,'Synchronized component means sum to end-to-end warm means. Processing/transfers are shown as an additional component so their cost is not silently assigned to image encoding. Loading, target disk reads and training are separate; support images are reprocessed without prefix caching. Timing hooks add synchronization overhead.')

if CFG['mode']=='study': rq7()


## 14 Export the complete analysis ZIP

In study mode the cell verifies all 14 PDF figures and 14 numbered CSV tables before creating the ZIP. It also saves prediction CSVs/JSONL, analysis contrasts, bin counts, captions, audit/split records, environment metadata and checksums. In pilot mode it exports the feasibility outputs only, clearly labelled as development results.

Keep the full private Kaggle output if you want to resume from trained checkpoints. The analysis ZIP excludes source images and model/adapter weight files; it includes the checkpoint metadata and measured training logs. The implementation contains no paid API calls.

**Validation of this supplied notebook:** syntax and dependency-free analysis paths were checked locally; synthetic fixtures exercised metric identities, temperature scaling, grouped splits/bootstrap, failure denominators, all seven analysis sections, all 14 figures/tables, omitted-adapter behavior and ZIP generation. Synthetic fixtures and their outputs are not included. GPU loading, image inference and adapter fitting were not executed in the authoring workspace; the Kaggle pilot is required to establish those capabilities.


In [ ]:
def finalize_outputs():
    status=dict(mode=CFG['mode'],run_contract=RUN_CONTRACT,
        created_utc=datetime.now(timezone.utc).isoformat(),
        grouping='source-confirmed participant mapping' if CFG['grouping_verified'] else 'provisional prefix; engineering benchmark',
        claims=['Current radiographic KL grading only; no diagnosis or patient-care validation.',
                'Foundation pretraining exposure to this public benchmark cannot be excluded.',
                'Predictions, scores, failure denominators and actual omissions are retained.'])
    if CFG['mode']=='study':
        expected_figures={f'Figure_{i:02d}.pdf' for i in range(1,15)}
        expected_tables={f'Table_{i:02d}.csv' for i in range(5,19)}
        present_figures={p.name for p in (ROOT/'figures').glob('*.pdf')}
        present_tables={p.name for p in (ROOT/'tables').glob('Table_*.csv')}
        status.update(missing_figures=sorted(expected_figures-present_figures),missing_tables=sorted(expected_tables-present_tables),
                      omitted=FROZEN['omitted'])
        assert not status['missing_figures'] and not status['missing_tables'], 'Incomplete requested outputs; inspect RQ cells.'
    else:
        status['next_step']='Review pilot and gradient gate; use a new study run with mode=study. No final-test results exist in this pilot.'
    dump_json(ROOT/'RUN_STATUS.json',status)
    readme=(
        'Knee osteoarthritis direct multimodal grading\n\n'
        f'Mode: {CFG["mode"]}\nGrouping: {status["grouping"]}\n\n'
        'This archive contains measured outputs from this run only. The proposal dummy values are never used.\n'
        'Figures: vector PDF plus notebook PNG. Tables: CSV. Captions: figures/captions.json.\n'
        'Protocol: manifests, fixed prompts, model revision, environment, scoring rule, temperatures and thresholds.\n'
        'Prediction rows retain failures; only complete predictions enter grade/calibration metrics.\n'
        'All-case accuracy and retained coverage use all eligible cases. Paired contrasts use common-complete cases.\n'
        'QWK differences are P0 minus full CNN for the primary contrast. Secondary test families use Holm.\n'
        'Bootstrap units are provisional prefixes unless authoritative participant mapping was documented.\n'
        'No clinical independence, external generalisation, feature explanation validity or superiority is assumed.\n'
        'Training time is measured wall time on the allocated GPU, not device-utilisation GPU-hours.\n'
        'Timing includes synchronization instrumentation; component means are instrumented estimates.\n'
        'No source images, pretrained base weights, or adapter weights are redistributed in this analysis archive.\n'
        'CNN checkpoints and LoRA adapter files remain in the Kaggle working directory. Preserve them privately\n'
        'as Kaggle output for resume; redistribution requires compliance with the respective source terms.\n'
        'For cross-session resume, mount the prior output directory and set resume_from to its pilot/study directory.\n'
    )
    (ROOT/'README.txt').write_text(readme,encoding='utf-8')
    # Include all analysis outputs and small training metadata, but do not bundle licensed
    # pretrained or adapted model weights, source images, caches, or large checkpoints.
    allowed=[]
    for f in sorted(ROOT.rglob('*')):
        if not f.is_file(): continue
        if f.suffix.lower() in {'.pt','.pth','.bin','.safetensors'}: continue
        allowed.append(f)
    dump_json(ROOT/'OUTPUT_CHECKSUMS.json',{str(f.relative_to(ROOT)):file_hash(f) for f in allowed
                                           if f.name!='OUTPUT_CHECKSUMS.json'})
    archive=ROOT.parent/f'Knee_OA_{CFG["mode"]}_outputs.zip'
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
        for f in sorted(ROOT.rglob('*')):
            if f.is_file() and f.suffix.lower() not in {'.pt','.pth','.bin','.safetensors'}:
                z.write(f,arcname=f.relative_to(ROOT))
    with zipfile.ZipFile(archive) as z: assert z.testzip() is None
    display(Markdown(f'**{CFG["mode"].title()} run complete. Download the output ZIP below.**'))
    # Link relative to the notebook server root, avoiding a duplicated /kaggle/working prefix.
    try:
        download_path=str(archive.relative_to(Path('/kaggle/working')))
    except ValueError:
        download_path=str(archive)
    display(FileLink(download_path))
    print('Archive size MiB:',round(archive.stat().st_size/2**20,2))
    print('Keep the whole Kaggle output privately if you need trained model checkpoints for resume.')
    return archive

OUTPUT_ZIP=finalize_outputs()


## Implementation references

The attached proposal is the experimental specification. Official implementation references consulted for the pinned backend:

- [Qwen2.5-VL model documentation, Transformers 4.51.3](https://huggingface.co/docs/transformers/v4.51.3/model_doc/qwen2_5_vl)
- [Qwen2.5-VL-3B-Instruct model card and files](https://huggingface.co/Qwen/Qwen2.5-VL-3B-Instruct)
- [PEFT quantized-model training documentation](https://huggingface.co/docs/peft/v0.15.0/developer_guides/quantization)
- [Versioned Qwen implementation source](https://github.com/huggingface/transformers/blob/v4.51.3/src/transformers/models/qwen2_5_vl/modeling_qwen2_5_vl.py)
- [Working Kaggle archive](https://www.kaggle.com/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity)

Do not equate deterministic schema validity, prompt stability, calibration improvements or review selection with radiographic correctness or clinical safety. This single public benchmark does not establish independent-hospital generalization, annotation certainty, pretraining non-contamination or superiority of either modeling strategy.

- [bitsandbytes 0.49.2 installation and CUDA build list](https://huggingface.co/docs/bitsandbytes/v0.49.2/installation)
